# PYNQ 301 - Memory Game Grid Detection

This notebook uses the YOLO object detection setup from `PYNQ 301 - Object Detection` to help build a memory game player.

The goal is to use one camera to detect two objects, draw boxes around the objects without drawing their class names, map each object into a board grid position, and write the object description plus grid location into memory.

## 1. Setup

Run these cells from the `PYNQ 301 - Object Detection` folder if possible. The notebook expects the same local model files used by the 301 object detection notebook:

- `tf_yolov3_voc.xmodel`
- `img/voc_classes.txt`

The grid mapping treats the camera image like a structured MNIST-style board: the image is divided into cells, and each detected object is assigned to the cell containing the center of its bounding box.

In [1]:

!pwd

/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/PYNQ 301 - Object Detection


In [2]:
from pathlib import Path
import os
import sys
import json
import re
import base64
import time
import requests
import random
import colorsys
import threading
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from IPython.display import display, Image
from pynq_dpu import DpuOverlay

%matplotlib inline

BOOTCAMP_301_DIR = Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/PYNQ 301 - Object Detection')
NOTEBOOK_DIR = Path.cwd()

if not (NOTEBOOK_DIR / 'tf_yolov3_voc.xmodel').exists() and BOOTCAMP_301_DIR.exists():
    NOTEBOOK_DIR = BOOTCAMP_301_DIR

MODEL_PATH = NOTEBOOK_DIR / 'tf_yolov3_voc.xmodel'
CLASSES_PATH = NOTEBOOK_DIR / 'img' / 'voc_classes.txt'

# The AI Riddle Hint feature reuses the shared halo Strix LLM helper from the
# ai_llm bootcamp notebook (server IP, model names, and chat plumbing all live there).
AI_HELPER_DIR = Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/ai_llm')
if str(AI_HELPER_DIR) not in sys.path:
    sys.path.append(str(AI_HELPER_DIR))
import bootcamp_ai

print(f'Using notebook assets from: {NOTEBOOK_DIR}')

Using notebook assets from: /home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/PYNQ 301 - Object Detection


In [3]:
overlay = DpuOverlay('dpu.bit')
overlay.load_model(str(MODEL_PATH))

RuntimeError: Programming Device failed: EBUSY (16) Device or resource busy/Bitstream in use by another program

## Optional OLED Adapter Test

Run this cell after the overlay is loaded to verify that a Grove I2C OLED can be reached through the Pmod Grove adapter. The default matches the related bootcamp OLED notebook: `PMODA`, port `G4`.

In [4]:
from pynq_peripherals import PmodGroveAdapter

# The Grove OLED driver talks to fixed I2C address 0x3c.
# Probe with raw I2C first because Gx='grove_oled' initializes the display in
# the constructor and can hang if SDA/SCL are wrong or the display is absent.
OLED_PMOD = overlay.PMODA
OLED_PORTS_TO_TRY = ('G4','G4')
OLED_ADDRESS = 0x3c
OLED_TEST_TEXT = 'OLED adapter OK'


def probe_oled_port(pmod, port, address=OLED_ADDRESS):
    adapter = PmodGroveAdapter(pmod, **{port: 'i2c'})
    i2c_dev = getattr(adapter, port)

    # SSD1306 command-mode prefix plus Display ON. A successful write should
    # return 2 bytes written; a missing device usually returns 0 or raises.
    # PYNQ's MicroBlaze RPC expects a mutable bytearray for buffer arguments.
    buffer = bytearray([0x80, 0xAF])
    written = adapter._lib.i2c_write(i2c_dev, address, buffer, len(buffer))
    adapter._lib.i2c_close(i2c_dev)
    return written


for port in OLED_PORTS_TO_TRY:
    print(f'Probing Grove OLED address 0x{OLED_ADDRESS:02x} on {OLED_PMOD["name"]} / {port}...')
    try:
        written = probe_oled_port(OLED_PMOD, port)
        print(f'  i2c_write returned {written}')
    except Exception as exc:
        print(f'  probe failed: {type(exc).__name__}: {exc}')

# Only run this after the raw probe returns 2 on a port. If this hangs, interrupt
# the kernel and check the adapter wiring/SDA/SCL route for that Grove port.
# OLED_PORT = 'G4'
# adapter = PmodGroveAdapter(OLED_PMOD, **{OLED_PORT: 'grove_oled'})
# oled = getattr(adapter, OLED_PORT)
# oled.set_normal_display()
# oled.clear_display()
# oled.put_string(OLED_TEST_TEXT)

NameError: name 'overlay' is not defined

In [ ]:
from pynq_peripherals import PmodGroveAdapter
import time

LED_BAR_PMOD = overlay.PMODA
LED_BAR_PORT = 'G4'
LED_BAR_BRIGHTNESS = 3
LED_BAR_GREEN_TO_RED = 1

ledbar_adapter = PmodGroveAdapter(LED_BAR_PMOD, **{LED_BAR_PORT: 'grove_ledbar'})
ledbar = getattr(ledbar_adapter, LED_BAR_PORT)

for level in range(1, 11):
    print(f'Setting LED Bar level to {level}')
    ledbar.set_level(level, LED_BAR_BRIGHTNESS, LED_BAR_GREEN_TO_RED)
    time.sleep(0.15)

time.sleep(0.5)

print('LED Bar test complete.')

## 2. YOLO Model Helpers

These helpers follow the same YOLO flow as the PYNQ 301 object detection notebook. The display code later draws only the bounding boxes, but the class description is still kept in memory for the game player.

In [5]:
def get_class(classes_path):
    with open(classes_path) as f:
        class_names = f.readlines()
    return [c.strip() for c in class_names]


class_names = get_class(CLASSES_PATH)

# Lower this if the turn debug image shows no boxes even when objects are visible.
YOLO_SCORE_THRESHOLD = 0.2

anchor_list = [10, 13, 16, 30, 33, 23, 30, 61, 62, 45, 59, 119, 116, 90, 156, 198, 373, 326]
anchor_float = [float(x) for x in anchor_list]
anchors = np.array(anchor_float).reshape(-1, 2)

num_classes = len(class_names)
hsv_tuples = [(1.0 * x / num_classes, 1.0, 1.0) for x in range(num_classes)]
colors = list(map(lambda x: colorsys.hsv_to_rgb(*x), hsv_tuples))
colors = list(map(lambda x: (int(x[0] * 255), int(x[1] * 255), int(x[2] * 255)), colors))
random.seed(0)
random.shuffle(colors)
random.seed(None)

In [6]:
def letterbox_image(image, size):
    ih, iw, _ = image.shape
    w, h = size
    scale = min(w / iw, h / ih)
    nw = int(iw * scale)
    nh = int(ih * scale)

    image = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_LINEAR)
    new_image = np.ones((h, w, 3), np.uint8) * 128
    h_start = (h - nh) // 2
    w_start = (w - nw) // 2
    new_image[h_start:h_start + nh, w_start:w_start + nw, :] = image
    return new_image


def pre_process(image, model_image_size):
    image = image[..., ::-1]
    image_h, image_w, _ = image.shape

    if model_image_size != (None, None):
        assert model_image_size[0] % 32 == 0, 'Multiples of 32 required'
        assert model_image_size[1] % 32 == 0, 'Multiples of 32 required'
        boxed_image = letterbox_image(image, tuple(reversed(model_image_size)))
    else:
        new_image_size = (image_w - (image_w % 32), image_h - (image_h % 32))
        boxed_image = letterbox_image(image, new_image_size)

    image_data = np.array(boxed_image, dtype='float32')
    image_data /= 255.0
    image_data = np.expand_dims(image_data, 0)
    return image_data


def _get_feats(feats, anchors, num_classes, input_shape):
    num_anchors = len(anchors)
    anchors_tensor = np.reshape(np.array(anchors, dtype=np.float32), [1, 1, 1, num_anchors, 2])
    grid_size = np.shape(feats)[1:3]
    nu = num_classes + 5
    predictions = np.reshape(feats, [-1, grid_size[0], grid_size[1], num_anchors, nu])
    grid_y = np.tile(np.reshape(np.arange(grid_size[0]), [-1, 1, 1, 1]), [1, grid_size[1], 1, 1])
    grid_x = np.tile(np.reshape(np.arange(grid_size[1]), [1, -1, 1, 1]), [grid_size[0], 1, 1, 1])
    grid = np.concatenate([grid_x, grid_y], axis=-1)
    grid = np.array(grid, dtype=np.float32)

    box_xy = (1 / (1 + np.exp(-predictions[..., :2])) + grid) / np.array(grid_size[::-1], dtype=np.float32)
    box_wh = np.exp(predictions[..., 2:4]) * anchors_tensor / np.array(input_shape[::-1], dtype=np.float32)
    box_confidence = 1 / (1 + np.exp(-predictions[..., 4:5]))
    box_class_probs = 1 / (1 + np.exp(-predictions[..., 5:]))
    return box_xy, box_wh, box_confidence, box_class_probs


def correct_boxes(box_xy, box_wh, input_shape, image_shape):
    box_yx = box_xy[..., ::-1]
    box_hw = box_wh[..., ::-1]
    input_shape = np.array(input_shape, dtype=np.float32)
    image_shape = np.array(image_shape, dtype=np.float32)
    new_shape = np.around(image_shape * np.min(input_shape / image_shape))
    offset = (input_shape - new_shape) / 2.0 / input_shape
    scale = input_shape / new_shape
    box_yx = (box_yx - offset) * scale
    box_hw *= scale

    box_mins = box_yx - (box_hw / 2.0)
    box_maxes = box_yx + (box_hw / 2.0)
    boxes = np.concatenate([
        box_mins[..., 0:1],
        box_mins[..., 1:2],
        box_maxes[..., 0:1],
        box_maxes[..., 1:2],
    ], axis=-1)
    boxes *= np.concatenate([image_shape, image_shape], axis=-1)
    return boxes


def boxes_and_scores(feats, anchors, classes_num, input_shape, image_shape):
    box_xy, box_wh, box_confidence, box_class_probs = _get_feats(feats, anchors, classes_num, input_shape)
    boxes = correct_boxes(box_xy, box_wh, input_shape, image_shape)
    boxes = np.reshape(boxes, [-1, 4])
    box_scores = box_confidence * box_class_probs
    box_scores = np.reshape(box_scores, [-1, classes_num])
    return boxes, box_scores


def nms_boxes(boxes, scores):
    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    areas = (x2 - x1 + 1) * (y2 - y1 + 1)
    order = scores.argsort()[::-1]
    keep = []

    while order.size > 0:
        i = order[0]
        keep.append(i)

        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w1 = np.maximum(0.0, xx2 - xx1 + 1)
        h1 = np.maximum(0.0, yy2 - yy1 + 1)
        inter = w1 * h1
        ovr = inter / (areas[i] + areas[order[1:]] - inter)
        inds = np.where(ovr <= 0.55)[0]
        order = order[inds + 1]

    return keep


def evaluate(yolo_outputs, image_shape, class_names, anchors, score_thresh=None):
    if score_thresh is None:
        score_thresh = YOLO_SCORE_THRESHOLD

    anchor_mask = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]
    boxes = []
    box_scores = []
    input_shape = np.shape(yolo_outputs[0])[1:3]
    input_shape = np.array(input_shape) * 32

    for i in range(len(yolo_outputs)):
        _boxes, _box_scores = boxes_and_scores(
            yolo_outputs[i], anchors[anchor_mask[i]], len(class_names), input_shape, image_shape
        )
        boxes.append(_boxes)
        box_scores.append(_box_scores)

    boxes = np.concatenate(boxes, axis=0)
    box_scores = np.concatenate(box_scores, axis=0)
    mask = box_scores >= score_thresh
    boxes_ = []
    scores_ = []
    classes_ = []

    for c in range(len(class_names)):
        class_boxes_np = boxes[mask[:, c]]
        class_box_scores_np = box_scores[:, c]
        class_box_scores_np = class_box_scores_np[mask[:, c]]
        nms_index_np = nms_boxes(class_boxes_np, class_box_scores_np)
        class_boxes_np = class_boxes_np[nms_index_np]
        class_box_scores_np = class_box_scores_np[nms_index_np]
        classes_np = np.ones_like(class_box_scores_np, dtype=np.int32) * c
        boxes_.append(class_boxes_np)
        scores_.append(class_box_scores_np)
        classes_.append(classes_np)

    if not boxes_:
        return np.array([]), np.array([]), np.array([])

    boxes_ = np.concatenate(boxes_, axis=0)
    scores_ = np.concatenate(scores_, axis=0)
    classes_ = np.concatenate(classes_, axis=0)
    return boxes_, scores_, classes_

In [7]:
dpu = overlay.runner
inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

shapeIn = tuple(inputTensors[0].dims)
shapeOut0 = tuple(outputTensors[0].dims)
shapeOut1 = tuple(outputTensors[1].dims)
shapeOut2 = tuple(outputTensors[2].dims)

input_data = [np.empty(shapeIn, dtype=np.float32, order='C')]
output_data = [
    np.empty(shapeOut0, dtype=np.float32, order='C'),
    np.empty(shapeOut1, dtype=np.float32, order='C'),
    np.empty(shapeOut2, dtype=np.float32, order='C'),
]
image = input_data[0]


def run(frame, score_thresh=None):
    image_size = frame.shape[:2]
    image_data = np.array(pre_process(frame, (416, 416)), dtype=np.float32)

    image[0, ...] = image_data.reshape(shapeIn[1:])
    job_id = dpu.execute_async(input_data, output_data)
    dpu.wait(job_id)

    conv_out0 = np.reshape(output_data[0], shapeOut0)
    conv_out1 = np.reshape(output_data[1], shapeOut1)
    conv_out2 = np.reshape(output_data[2], shapeOut2)
    yolo_outputs = [conv_out0, conv_out1, conv_out2]

    return evaluate(yolo_outputs, image_size, class_names, anchors, score_thresh=score_thresh)

NameError: name 'overlay' is not defined

## 3. Grid Mapping

Set `GRID_ROWS` and `GRID_COLS` to match the memory game board.

If the camera is pointed straight at the board, leave `BOARD_CORNERS = None`. If the camera is angled, set `BOARD_CORNERS` to four pixel coordinates in this order:

`top-left`, `top-right`, `bottom-right`, `bottom-left`

The grid position is calculated from the center of each YOLO bounding box.

In [8]:
GRID_ROWS = 5
GRID_COLS = 6

# Example for an angled camera:
# BOARD_CORNERS = np.float32([[80, 60], [1180, 80], [1220, 690], [60, 700]])
BOARD_CORNERS = None


def board_source_corners(frame_shape):
    height, width = frame_shape[:2]
    if BOARD_CORNERS is not None:
        return np.float32(BOARD_CORNERS)
    return np.float32([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1],
    ])


def board_transform(frame_shape):
    source = board_source_corners(frame_shape)
    destination = np.float32([
        [0, 0],
        [GRID_COLS, 0],
        [GRID_COLS, GRID_ROWS],
        [0, GRID_ROWS],
    ])
    return cv2.getPerspectiveTransform(source, destination)


def frame_transform(frame_shape):
    source = np.float32([
        [0, 0],
        [GRID_COLS, 0],
        [GRID_COLS, GRID_ROWS],
        [0, GRID_ROWS],
    ])
    destination = board_source_corners(frame_shape)
    return cv2.getPerspectiveTransform(source, destination)


def box_center(box):
    y_min, x_min, y_max, x_max = map(float, box)
    return (x_min + x_max) / 2.0, (y_min + y_max) / 2.0


def grid_position_for_box(box, frame_shape):
    center_x, center_y = box_center(box)
    point = np.float32([[[center_x, center_y]]])
    grid_x, grid_y = cv2.perspectiveTransform(point, board_transform(frame_shape))[0][0]

    col = int(np.clip(np.floor(grid_x), 0, GRID_COLS - 1))
    row = int(np.clip(np.floor(grid_y), 0, GRID_ROWS - 1))
    return {
        'row': row,
        'col': col,
        'cell': f'R{row + 1}C{col + 1}',
        'center': (float(center_x), float(center_y)),
    }


def draw_grid(frame):
    frame = frame.copy()
    to_frame = frame_transform(frame.shape)

    for col in range(GRID_COLS + 1):
        points = np.float32([[[col, 0]], [[col, GRID_ROWS]]])
        p1, p2 = cv2.perspectiveTransform(points, to_frame).reshape(2, 2).astype(int)
        cv2.line(frame, tuple(p1), tuple(p2), (0, 255, 0), 1)

    for row in range(GRID_ROWS + 1):
        points = np.float32([[[0, row]], [[GRID_COLS, row]]])
        p1, p2 = cv2.perspectiveTransform(points, to_frame).reshape(2, 2).astype(int)
        cv2.line(frame, tuple(p1), tuple(p2), (0, 255, 0), 1)

    return frame


## 4. Object Memory

The screen shows only boxes and grid lines. The object class name is not drawn on the camera image, but it is saved in `object_memory` for the game player.

In [9]:
MAX_OBJECTS = 2
object_memory = {}


def empty_object_memory():
    global object_memory
    object_memory = {}
    return object_memory


## 5. Open One Camera

This scans `/dev/video*` and opens the first camera that can return a frame, so the notebook does not depend on a hard-coded video id.

In [10]:
CAMERA_SIZE = (1920, 1080)
CAMERA_FPS = 15
CAMERA_BUFFER_SIZE = 1
CAMERA_VERTICAL_FLIP = False  # flip upside-down board camera frames right-side up


def set_camera_vertical_flip(enabled):
    global CAMERA_VERTICAL_FLIP
    CAMERA_VERTICAL_FLIP = bool(enabled)
    print(f'Camera vertical flip: {CAMERA_VERTICAL_FLIP}')
    return CAMERA_VERTICAL_FLIP


def video_devices():
    devices = []
    for name in os.listdir('/dev'):
        if name.startswith('video') and name[5:].isdigit():
            devices.append((int(name[5:]), f'/dev/{name}'))
    return [device for _, device in sorted(devices)]


def open_camera(device):
    camera = cv2.VideoCapture(device, cv2.CAP_V4L2)
    camera.set(cv2.CAP_PROP_BUFFERSIZE, CAMERA_BUFFER_SIZE)
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, CAMERA_SIZE[0])
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_SIZE[1])
    camera.set(cv2.CAP_PROP_FPS, CAMERA_FPS)
    return camera


def scan_working_camera():
    for device in video_devices():
        camera = open_camera(device)
        ret, _ = camera.read()
        if camera.isOpened() and ret:
            print(f'Using camera from {device}')
            return camera, device
        camera.release()
    raise RuntimeError('Could not find a working /dev/video* camera.')


if 'cap' in globals():
    cap.release()

cap, camera_device = scan_working_camera()
display_handle = display(None, display_id=True)

Using camera from /dev/video0


None

## 6. Camera Alignment Test Loop

Run this before starting the game to position the camera and tune `BOARD_CORNERS`, `GRID_ROWS`, and `GRID_COLS`.

By default, the loop only draws the grid so it stays responsive. Set `run_detection=True` if you also want to see the YOLO object boxes while aligning the board.

Interrupt the cell when the camera position looks correct.

In [11]:
def draw_camera_test_detections(frame, boxes, scores, classes):
    frame = draw_grid(frame)
    if not scores.any():
        return cv2.putText(
            frame,
            'No YOLO detections',
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 255),
            2,
            cv2.LINE_AA,
        )

    top_n = min(MAX_OBJECTS, len(scores))
    top_indices = np.argsort(scores)[-top_n:][::-1]
    for object_number, idx in enumerate(top_indices, start=1):
        y_min, x_min, y_max, x_max = map(int, boxes[idx])
        color = colors[int(classes[idx])] if len(colors) else (255, 255, 255)
        frame = cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), color, 2)
        grid_info = grid_position_for_box(boxes[idx], frame.shape)
        label = f"{object_number}: {class_names[int(classes[idx])]} {scores[idx]:.2f} {grid_info['cell']}"
        frame = cv2.putText(
            frame,
            label,
            (x_min, max(y_min - 10, 30)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2,
            cv2.LINE_AA,
        )

    return frame


def camera_test_loop(run_detection=False, flush_frames=1):
    """Continuously show the camera feed with the grid overlay for setup."""
    test_display_handle = display(None, display_id=True)

    try:
        while True:
            if 'capture_frame' in globals():
                frame = capture_frame(flush_frames=flush_frames, settle_seconds=0.0)
            else:
                frame = None
                for _ in range(max(1, flush_frames)):
                    ret, frame = cap.read()
                    if not ret:
                        frame = None
                if frame is None:
                    continue

            if run_detection:
                detection_frame = frame
                boxes, scores, classes = run(detection_frame)
                frame = draw_camera_test_detections(detection_frame, boxes, scores, classes)

                if 'records_for_current_mode' in globals() and 'annotate_records_frame' in globals():
                    records = records_for_current_mode(detection_frame)
                    frame = annotate_records_frame(frame, records, turn_label='Camera test')
            else:
                frame = draw_grid(frame)

            _, encoded_frame = cv2.imencode('.jpeg', frame)
            test_display_handle.update(Image(data=encoded_frame.tobytes()))
    except KeyboardInterrupt:
        print('Camera test loop stopped.')


# Run one of these manually when you are ready to align the camera:
#camera_test_loop()
# # # # # # # # # # # # # # # camera_test_loop(run_detection=True)


## 7. One-Shot Detection for a Turn

The game is turn-based, so this notebook should not sit inside an infinite camera loop.

Use `DETECTION_MODE` to choose the detector:

```python
set_detection_mode('grid')        # Crop/stretch each grid square before YOLO.
set_detection_mode('full_frame')  # Run YOLO once on the whole image, like the original notebook.
```

Use `detect_revealed_cards()` whenever the board has two revealed cards. It waits briefly for the board to settle, drains queued camera frames, captures one current frame, runs the selected detection mode, updates `object_memory`, and returns the detected cards.

The turn functions in the next section call `detect_revealed_cards()` automatically, so the normal game flow is:

1. Flip cards on the physical board.
2. Call `record_opponent_turn()` or `record_computer_turn_result()`.
3. The function records the two highest-confidence detections from the selected mode.

In [12]:
TURN_DEBUG_SHOW_LABELS = True
TURN_DEBUG_SHOW_ALL_DETECTIONS = False
TURN_CAPTURE_SETTLE_SECONDS = 0.4
TURN_CAPTURE_FLUSH_FRAMES = 12
TURN_CAPTURE_PHOTO_COUNT = 3
TURN_CAPTURE_PHOTO_INTERVAL_SECONDS = 0.15

# ArUco marker settings. Change this if your printed markers use a different
# OpenCV dictionary, such as 'DICT_5X5_100' or 'DICT_6X6_250'.
ARUCO_DICTIONARY_NAME = 'DICT_4X4_50'
# If every photo misses one of the two revealed objects, the same photos are
# retried at these lower thresholds before giving up. These only affect YOLO
# fallback code; ArUco marker detection ignores YOLO score thresholds.
YOLO_FALLBACK_SCORE_THRESHOLDS = [0.15, 0.1, 0.05]

GRID_CROP_SIZE = (416, 416)  # width x height for per-cell YOLO crops.

# ── Detection approach ────────────────────────────────────────────────────────
# Pick the strategy that matches how your cards and arena are set up.
# Use the widget dropdown or set this variable directly.
#
#   'yolo_full_frame' — YOLO runs on the whole camera frame. No ArUco markers
#                       needed. Set BOARD_CORNERS in the Grid Mapping cell if
#                       your camera is angled.
#
#   'yolo_grid_crops' — YOLO runs on each grid cell individually (each cell is
#                       cropped and stretched to 416x416 before inference).
#                       More robust when the background is cluttered.
#                       Set BOARD_CORNERS for angled cameras.
#
#   'aruco_border'    — Print markers 3-6 at the four physical corners of the
#                       arena (outside the card area). They auto-calibrate the
#                       board every turn so BOARD_CORNERS is never needed.
#                       YOLO detects objects. Markers are never occluded by
#                       face-up cards.
#
#   'aruco_per_card'  — Print markers 0-29 on the front (object side) of each
#                       card. The marker ID encodes the cell directly so no
#                       perspective math is needed. YOLO detects objects; ArUco
#                       identifies which cell the face-up card is in.
#
DETECTION_APPROACH = 'aruco_per_card'

# Internal variables kept for backward compatibility with helper functions.
# Set automatically by set_detection_approach(); do not edit directly.
DETECTION_MODE = 'full_frame'
CARD_POSITION_MODE = 'aruco_id'


def update_object_memory_from_grid_records(records):
    global object_memory
    object_memory = {}

    best_records = sorted(records, key=record_selection_key, reverse=True)[:MAX_OBJECTS]
    for object_number, record in enumerate(best_records, start=1):
        object_memory[f'object_{object_number}'] = {
            'description': record['description'],
            'score': float(record.get('score', 1.0)),
            'row': int(record['row']),
            'col': int(record['col']),
            'cell': record['cell'],
            'center': tuple(record['center']),
            'box': [float(value) for value in record['box']],
            'crop_box': [float(value) for value in record['crop_box']],
            'updated_at': time.time(),
        }
        if 'aruco_id' in record:
            object_memory[f'object_{object_number}']['aruco_id'] = int(record['aruco_id'])

    return object_memory


def record_selection_key(record):
    return (
        record.get('detector') == 'yolo',
        'aruco_id' in record,
        record.get('score', 1.0),
    )


def print_object_memory(memory):
    if not memory:
        print('No objects detected')
        return

    for name, info in memory.items():
        marker_text = f" id={info['aruco_id']}" if 'aruco_id' in info else ''
        print(f"{name}: {info['description']}{marker_text} at {info['cell']} score={info['score']:.2f}")


def set_detection_mode(mode):
    global DETECTION_MODE
    if mode not in ('grid', 'full_frame'):
        raise ValueError("mode must be 'grid' or 'full_frame'")
    DETECTION_MODE = mode
    return DETECTION_MODE


def set_card_position_mode(mode):
    global CARD_POSITION_MODE
    if mode not in ('yolo_center', 'aruco_id'):
        raise ValueError("mode must be 'yolo_center' or 'aruco_id'")
    CARD_POSITION_MODE = mode
    return CARD_POSITION_MODE


_APPROACH_SETTINGS = {
    'yolo_full_frame':        ('full_frame', 'yolo_center'),
    'yolo_grid_crops':        ('grid',       'yolo_center'),
    'aruco_border':           ('full_frame', 'yolo_center'),
    'aruco_per_card':         ('full_frame', 'aruco_id'),
    'aruco_border_grid_crops': ('grid',      'yolo_center'),
}


def set_detection_approach(approach):
    global DETECTION_APPROACH
    if approach not in _APPROACH_SETTINGS:
        raise ValueError(
            f"Unknown approach: {approach!r}. "
            f"Choose from: {list(_APPROACH_SETTINGS)}"
        )
    DETECTION_APPROACH = approach
    det_mode, pos_mode = _APPROACH_SETTINGS[approach]
    set_detection_mode(det_mode)
    set_card_position_mode(pos_mode)
    print(f'[detection] approach set to: {approach}')


def camera_is_open():
    return 'cap' in globals() and cap is not None and cap.isOpened()


def reopen_camera():
    global cap, camera_device
    if camera_is_open():
        cap.release()

    if 'camera_device' in globals():
        cap = open_camera(camera_device)
        for _ in range(5):
            ret, frame = cap.read()
            if ret and frame is not None:
                print(f'Reopened camera from {camera_device}')
                return cap
            time.sleep(0.05)
        cap.release()

    cap, camera_device = scan_working_camera()
    return cap


def ensure_camera_open():
    if not camera_is_open():
        reopen_camera()
    return cap


def drain_camera_buffer(flush_frames=TURN_CAPTURE_FLUSH_FRAMES):
    """Drop queued camera frames so turn detection sees the current board."""
    ensure_camera_open()
    for _ in range(max(0, flush_frames)):
        cap.grab()


def apply_camera_vertical_flip(frame):
    if frame is not None and CAMERA_VERTICAL_FLIP:
        return cv2.flip(frame, 0)
    return frame


def read_camera_frame(read_attempts=8):
    ensure_camera_open()
    for _ in range(max(1, int(read_attempts))):
        ret, frame = cap.read()
        if ret and frame is not None:
            return apply_camera_vertical_flip(frame)

        # Some V4L2 drivers need retrieve() after grab() instead of read().
        cap.grab()
        ret, frame = cap.retrieve()
        if ret and frame is not None:
            return apply_camera_vertical_flip(frame)

        time.sleep(0.05)

    return None


def capture_frame(flush_frames=TURN_CAPTURE_FLUSH_FRAMES, settle_seconds=TURN_CAPTURE_SETTLE_SECONDS, read_attempts=8):
    """Grab one fresh frame from the open camera after the board has settled."""
    if settle_seconds > 0:
        time.sleep(settle_seconds)

    for attempt in range(2):
        drain_camera_buffer(flush_frames if attempt == 0 else 0)
        frame = read_camera_frame(read_attempts=read_attempts)
        if frame is not None:
            return frame

        reopen_camera()

    raise RuntimeError(
        'Could not read a fresh frame from the camera. '
        'Check that no alignment loop is still running, the camera is connected, and /dev/video is not busy.'
    )



def grid_position_for_point(point, frame_shape):
    point = np.float32([[[float(point[0]), float(point[1])]]])
    grid_x, grid_y = cv2.perspectiveTransform(point, board_transform(frame_shape))[0][0]

    col = int(np.clip(np.floor(grid_x), 0, GRID_COLS - 1))
    row = int(np.clip(np.floor(grid_y), 0, GRID_ROWS - 1))
    return {
        'row': row,
        'col': col,
        'cell': f'R{row + 1}C{col + 1}',
        'center': (float(point[0][0][0]), float(point[0][0][1])),
    }


def grid_position_for_aruco_id(marker_id, fallback_center):
    marker_id = int(marker_id)
    cell_index = marker_id % (GRID_ROWS * GRID_COLS)
    row = cell_index // GRID_COLS
    col = cell_index % GRID_COLS
    return {
        'row': row,
        'col': col,
        'cell': f'R{row + 1}C{col + 1}',
        'center': tuple(map(float, fallback_center)),
    }


def aruco_grid_position(marker_id, center, frame_shape):
    if CARD_POSITION_MODE == 'aruco_id':
        return grid_position_for_aruco_id(marker_id, center)
    return grid_position_for_point(center, frame_shape)


def grid_cell_quad(row, col, frame_shape):
    to_frame = frame_transform(frame_shape)
    grid_points = np.float32([[
        [col, row],
        [col + 1, row],
        [col + 1, row + 1],
        [col, row + 1],
    ]])
    return cv2.perspectiveTransform(grid_points, to_frame).reshape(4, 2).astype(np.float32)


def crop_grid_cell(frame, row, col):
    width, height = GRID_CROP_SIZE
    src = grid_cell_quad(row, col, frame.shape)
    dst = np.float32([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1],
    ])
    transform = cv2.getPerspectiveTransform(src, dst)
    crop = cv2.warpPerspective(frame, transform, (width, height))
    return crop, src, dst


def crop_box_to_frame_box(crop_box, src_quad, dst_quad, frame_shape):
    y_min, x_min, y_max, x_max = map(float, crop_box)
    crop_corners = np.float32([[
        [x_min, y_min],
        [x_max, y_min],
        [x_max, y_max],
        [x_min, y_max],
    ]])
    inverse_transform = cv2.getPerspectiveTransform(dst_quad, src_quad)
    frame_corners = cv2.perspectiveTransform(crop_corners, inverse_transform).reshape(4, 2)

    frame_height, frame_width = frame_shape[:2]
    xs = np.clip(frame_corners[:, 0], 0, frame_width - 1)
    ys = np.clip(frame_corners[:, 1], 0, frame_height - 1)
    return [float(np.min(ys)), float(np.min(xs)), float(np.max(ys)), float(np.max(xs))]


def crop_box_center_to_frame(crop_box, src_quad, dst_quad):
    y_min, x_min, y_max, x_max = map(float, crop_box)
    crop_center = np.float32([[[((x_min + x_max) / 2.0), ((y_min + y_max) / 2.0)]]])
    inverse_transform = cv2.getPerspectiveTransform(dst_quad, src_quad)
    center = cv2.perspectiveTransform(crop_center, inverse_transform)[0][0]
    return (float(center[0]), float(center[1]))


def crop_points_to_frame(points, src_quad, dst_quad):
    inverse_transform = cv2.getPerspectiveTransform(dst_quad, src_quad)
    frame_points = cv2.perspectiveTransform(np.float32([points]), inverse_transform)[0]
    return frame_points.astype(np.float32)


def aruco_dictionary():
    if not hasattr(cv2, 'aruco'):
        raise RuntimeError('OpenCV ArUco support is not available. Install opencv-contrib-python or use an OpenCV build with cv2.aruco.')
    dictionary_id = getattr(cv2.aruco, ARUCO_DICTIONARY_NAME, None)
    if dictionary_id is None:
        raise ValueError(f'Unknown ArUco dictionary: {ARUCO_DICTIONARY_NAME}')
    return cv2.aruco.getPredefinedDictionary(dictionary_id)


def aruco_detector_parameters():
    if hasattr(cv2.aruco, 'DetectorParameters'):
        return cv2.aruco.DetectorParameters()
    return cv2.aruco.DetectorParameters_create()


def detect_aruco_corners_and_ids(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    dictionary = aruco_dictionary()
    parameters = aruco_detector_parameters()

    if hasattr(cv2.aruco, 'ArucoDetector'):
        detector = cv2.aruco.ArucoDetector(dictionary, parameters)
        corners, ids, rejected = detector.detectMarkers(gray)
    else:
        corners, ids, rejected = cv2.aruco.detectMarkers(gray, dictionary, parameters=parameters)

    if ids is None:
        return [], np.array([], dtype=np.int32)
    return corners, ids.flatten().astype(np.int32)


# ArUco IDs reserved for the four arena corner markers (aruco_border approach).
# Print these outside the card grid so they are never covered by face-up cards.
BORDER_MARKER_TL = 3  # top-left corner
BORDER_MARKER_TR = 4  # top-right corner
BORDER_MARKER_BR = 5  # bottom-right corner
BORDER_MARKER_BL = 6  # bottom-left corner


def calibrate_from_border_markers(frame):
    """Detect the 4 arena corner markers (BORDER_MARKER_TL/TR/BR/BL) and update BOARD_CORNERS.

    Returns the number of border markers found. BOARD_CORNERS is only updated
    when all 4 are visible. Call once before detection each turn in aruco_border
    mode — handled automatically by records_for_current_mode.
    """
    global BOARD_CORNERS
    corners, ids = detect_aruco_corners_and_ids(frame)
    corner_map = {
        BORDER_MARKER_TL: 0,
        BORDER_MARKER_TR: 1,
        BORDER_MARKER_BR: 2,
        BORDER_MARKER_BL: 3,
    }
    found = {}
    for marker_corners, marker_id in zip(corners, ids):
        mid = int(marker_id)
        if mid in corner_map:
            found[corner_map[mid]] = np.mean(marker_corners.reshape(4, 2), axis=0)
    if len(found) == 4:
        BOARD_CORNERS = np.float32([found[0], found[1], found[2], found[3]])
    return len(found)



def aruco_records_for_frame(frame, detection_mode):
    """Detect ArUco markers and map every marker ID to a memory-game grid cell.

    Border markers (BORDER_MARKER_TL/TR/BR/BL) are explicitly excluded so
    they never appear as false card detections in aruco_per_card mode --
    checked by exact ID, not just a >= card_id_limit range, since the
    border IDs (3-6) now fall inside the card ID range (0..GRID_ROWS*GRID_COLS-1)
    instead of safely above it.
    """
    corners, ids = detect_aruco_corners_and_ids(frame)
    records = []
    border_marker_ids = {BORDER_MARKER_TL, BORDER_MARKER_TR, BORDER_MARKER_BR, BORDER_MARKER_BL}
    card_id_limit = GRID_ROWS * GRID_COLS

    for marker_corners, marker_id in zip(corners, ids):
        if int(marker_id) in border_marker_ids or int(marker_id) >= card_id_limit:
            continue
        points = marker_corners.reshape(4, 2).astype(np.float32)
        center = tuple(np.mean(points, axis=0).astype(float))
        grid_info = aruco_grid_position(marker_id, center, frame.shape)
        x_min, y_min = np.min(points, axis=0)
        x_max, y_max = np.max(points, axis=0)

        records.append({
            'description': f'aruco_{int(marker_id)}',
            'score': 1.0,
            'class_index': int(marker_id),
            'aruco_id': int(marker_id),
            'row': grid_info['row'],
            'col': grid_info['col'],
            'cell': grid_info['cell'],
            'center': grid_info['center'],
            'corners': [[float(x), float(y)] for x, y in points],
            'box': [float(y_min), float(x_min), float(y_max), float(x_max)],
            'crop_box': [float(y_min), float(x_min), float(y_max), float(x_max)],
            'detection_mode': detection_mode,
            'detector': 'aruco',
        })

    return sorted(records, key=lambda record: (record['row'], record['col'], record['aruco_id']))


def aruco_records_for_grid_cell(frame, row, col):
    """Crop one grid cell, detect ArUco markers in that stretched crop, and map them back to the frame."""
    crop, src_quad, dst_quad = crop_grid_cell(frame, row, col)
    corners, ids = detect_aruco_corners_and_ids(crop)
    records = []

    for marker_corners, marker_id in zip(corners, ids):
        crop_points = marker_corners.reshape(4, 2).astype(np.float32)
        frame_points = crop_points_to_frame(crop_points, src_quad, dst_quad)
        center = tuple(np.mean(frame_points, axis=0).astype(float))
        grid_info = aruco_grid_position(marker_id, center, frame.shape)
        crop_x_min, crop_y_min = np.min(crop_points, axis=0)
        crop_x_max, crop_y_max = np.max(crop_points, axis=0)
        frame_x_min, frame_y_min = np.min(frame_points, axis=0)
        frame_x_max, frame_y_max = np.max(frame_points, axis=0)

        records.append({
            'description': f'aruco_{int(marker_id)}',
            'score': 1.0,
            'class_index': int(marker_id),
            'aruco_id': int(marker_id),
            'row': grid_info['row'],
            'col': grid_info['col'],
            'cell': grid_info['cell'],
            'center': grid_info['center'],
            'corners': [[float(x), float(y)] for x, y in frame_points],
            'box': [float(frame_y_min), float(frame_x_min), float(frame_y_max), float(frame_x_max)],
            'crop_box': [float(crop_y_min), float(crop_x_min), float(crop_y_max), float(crop_x_max)],
            'detection_mode': 'grid',
            'detector': 'aruco',
        })

    return records


def yolo_record_from_box(frame_shape, box, score, class_index, detection_mode, crop_box=None):
    y_min, x_min, y_max, x_max = map(float, box)
    center = ((x_min + x_max) / 2.0, (y_min + y_max) / 2.0)
    grid_info = grid_position_for_point(center, frame_shape)
    return {
        'description': class_names[int(class_index)],
        'score': float(score),
        'class_index': int(class_index),
        'row': grid_info['row'],
        'col': grid_info['col'],
        'cell': grid_info['cell'],
        'center': grid_info['center'],
        'box': [y_min, x_min, y_max, x_max],
        'crop_box': [float(value) for value in (crop_box if crop_box is not None else box)],
        'detection_mode': detection_mode,
        'detector': 'yolo',
    }


def best_detection_for_grid_cell(frame, row, col, score_thresh=None):
    """Run YOLO on one perspective-corrected grid cell and map the best box back to the frame."""
    crop, src_quad, dst_quad = crop_grid_cell(frame, row, col)
    boxes, scores, classes = run(crop, score_thresh=score_thresh)
    if not scores.any():
        return None

    best_idx = int(np.argmax(scores))
    crop_box = boxes[best_idx]
    frame_box = crop_box_to_frame_box(crop_box, src_quad, dst_quad, frame.shape)
    record = yolo_record_from_box(
        frame.shape,
        frame_box,
        scores[best_idx],
        classes[best_idx],
        detection_mode='grid',
        crop_box=crop_box,
    )
    record['row'] = int(row)
    record['col'] = int(col)
    record['cell'] = f'R{row + 1}C{col + 1}'
    record['center'] = crop_box_center_to_frame(crop_box, src_quad, dst_quad)
    return record


def yolo_records_for_grid(frame, score_thresh=None):
    records = []
    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            record = best_detection_for_grid_cell(frame, row, col, score_thresh=score_thresh)
            if record is not None:
                records.append(record)
    return sorted(records, key=lambda record: record['score'], reverse=True)


def aruco_records_for_grid(frame):
    records = []
    for row in range(GRID_ROWS):
        for col in range(GRID_COLS):
            records.extend(aruco_records_for_grid_cell(frame, row, col))
    return sorted(records, key=lambda record: (record['row'], record['col'], record['aruco_id']))


def yolo_records_for_full_frame(frame, score_thresh=None):
    boxes, scores, classes = run(frame, score_thresh=score_thresh)
    if not scores.any():
        return []
    records = [
        yolo_record_from_box(frame.shape, box, score, class_index, detection_mode='full_frame')
        for box, score, class_index in zip(boxes, scores, classes)
    ]
    return sorted(records, key=lambda record: record['score'], reverse=True)


def merge_aruco_with_yolo_records(yolo_records, aruco_records):
    """Attach nearest ArUco IDs to YOLO cards, optionally using marker IDs for stored positions."""
    merged_records = [dict(record) for record in yolo_records]
    used_aruco_indices = set()

    for yolo_record in merged_records:
        if not aruco_records:
            break

        yolo_center = np.array(yolo_record['center'], dtype=np.float32)
        candidate_indices = [
            index for index in range(len(aruco_records))
            if index not in used_aruco_indices
        ]
        if not candidate_indices:
            break

        best_index = min(
            candidate_indices,
            key=lambda index: np.linalg.norm(np.array(aruco_records[index]['center'], dtype=np.float32) - yolo_center),
        )
        aruco_record = aruco_records[best_index]
        aruco_center = tuple(map(float, aruco_record['center']))

        yolo_record['yolo_cell'] = yolo_record['cell']
        yolo_record['yolo_center'] = yolo_record['center']
        yolo_record['aruco_id'] = int(aruco_record['aruco_id'])
        yolo_record['aruco_score'] = float(aruco_record.get('score', 1.0))
        yolo_record['aruco_center'] = aruco_center
        yolo_record['corners'] = aruco_record['corners']

        if CARD_POSITION_MODE == 'aruco_id':
            grid_info = grid_position_for_aruco_id(aruco_record['aruco_id'], aruco_center)
            yolo_record['row'] = int(grid_info['row'])
            yolo_record['col'] = int(grid_info['col'])
            yolo_record['cell'] = grid_info['cell']

        used_aruco_indices.add(best_index)

    unmatched_aruco_records = [
        dict(aruco_record)
        for index, aruco_record in enumerate(aruco_records)
        if index not in used_aruco_indices
    ]
    merged_records.extend(unmatched_aruco_records)
    return sorted(merged_records, key=record_selection_key, reverse=True)


def point_inside_box(point, box):
    y_min, x_min, y_max, x_max = map(float, box)
    x, y = map(float, point)
    return x_min <= x <= x_max and y_min <= y <= y_max


def scan_grid_cells_for_objects(frame, score_thresh=None):
    """Crop/stretch each grid cell, then run YOLO and ArUco marker detection on each crop."""
    yolo_records = yolo_records_for_grid(frame, score_thresh=score_thresh)
    aruco_records = aruco_records_for_grid(frame)
    return merge_aruco_with_yolo_records(yolo_records, aruco_records)


def full_frame_records(frame, score_thresh=None):
    """Run YOLO and ArUco marker detection on the full camera frame."""
    yolo_records = yolo_records_for_full_frame(frame, score_thresh=score_thresh)
    aruco_records = aruco_records_for_frame(frame, detection_mode='full_frame')
    return merge_aruco_with_yolo_records(yolo_records, aruco_records)


def records_for_current_mode(frame, score_thresh=None):
    """Detect face-up cards using the configured DETECTION_APPROACH.

    'yolo_full_frame' — YOLO on the whole frame; set BOARD_CORNERS for angled cameras.
    'yolo_grid_crops' — YOLO on per-cell crops; best for cluttered backgrounds.
    'aruco_border'    — 4 corner markers (IDs 3-6) auto-calibrate the board; YOLO finds objects.
    'aruco_per_card'  — per-card markers (IDs 0-29) identify cells directly; YOLO finds objects.
    'aruco_border_grid_crops' — same corner auto-calibration as aruco_border, but per-cell crops like yolo_grid_crops instead of full-frame.
    """
    approach = globals().get('DETECTION_APPROACH', 'aruco_per_card')

    if approach == 'yolo_full_frame':
        return yolo_records_for_full_frame(frame, score_thresh=score_thresh)

    if approach == 'yolo_grid_crops':
        return scan_grid_cells_for_objects(frame, score_thresh=score_thresh)

    if approach == 'aruco_border':
        n = calibrate_from_border_markers(frame)
        if n < 4:
            print(f'[aruco_border] {n}/4 border markers visible — BOARD_CORNERS not updated')
        return yolo_records_for_full_frame(frame, score_thresh=score_thresh)

    if approach == 'aruco_per_card':
        yolo_records = yolo_records_for_full_frame(frame, score_thresh=score_thresh)
        aruco_records = aruco_records_for_frame(frame, detection_mode='full_frame')
        return merge_aruco_with_yolo_records(yolo_records, aruco_records)

    if approach == 'aruco_border_grid_crops':
        n = calibrate_from_border_markers(frame)
        if n < 4:
            print(f'[aruco_border_grid_crops] {n}/4 border markers visible — BOARD_CORNERS not updated')
        return scan_grid_cells_for_objects(frame, score_thresh=score_thresh)

    raise ValueError(
        f"Unknown DETECTION_APPROACH: {approach!r}. "
        "Choose from: 'yolo_full_frame', 'yolo_grid_crops', 'aruco_border', 'aruco_per_card', 'aruco_border_grid_crops'"
    )


def draw_detection_label(frame, label, x, y, color=(255, 255, 255)):
    text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
    y = max(y, text_size[1] + 8)
    cv2.rectangle(
        frame,
        (x, y - text_size[1] - 8),
        (x + text_size[0] + 8, y + 4),
        (0, 0, 0),
        -1,
    )
    return cv2.putText(frame, label, (x + 4, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)


def record_color(record):
    if not len(colors):
        return (255, 255, 255)
    return colors[int(record.get('class_index', 0)) % len(colors)]


def draw_aruco_marker(frame, record, color):
    points = np.array(record['corners'], dtype=np.int32)
    cv2.polylines(frame, [points], True, color, 2)
    marker_center = record.get('aruco_center', record['center'])
    center_x, center_y = map(int, marker_center)
    cv2.circle(frame, (center_x, center_y), 4, color, -1)
    return frame


def marker_label(record):
    if 'aruco_id' in record:
        if record.get('detector') == 'yolo':
            return f"{record['description']} {record['score']:.2f} ID {record['aruco_id']} {record['cell']}"
        return f"ID {record['aruco_id']} {record['cell']}"
    return f"{record['description']} {record['score']:.2f} {record['cell']}"


def annotate_records_frame(frame, records, turn_label='Current detection'):
    annotated = draw_grid(frame)
    sorted_records = sorted(records, key=lambda record: record.get('score', 1.0), reverse=True)
    debug_records = sorted_records if TURN_DEBUG_SHOW_ALL_DETECTIONS else sorted_records[:MAX_OBJECTS]

    for object_number, record in enumerate(debug_records, start=1):
        y_min, x_min, y_max, x_max = map(int, record['box'])
        color = record_color(record)

        if record.get('detector') == 'yolo':
            annotated = cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), color, 2)
        if 'corners' in record:
            annotated = draw_aruco_marker(annotated, record, color)
        elif record.get('detector') != 'yolo':
            annotated = cv2.rectangle(annotated, (x_min, y_min), (x_max, y_max), color, 2)

        if TURN_DEBUG_SHOW_LABELS:
            label = f"{object_number}: {marker_label(record)}"
            annotated = draw_detection_label(annotated, label, x_min, y_min, color)

    marker_count = len([record for record in records if 'aruco_id' in record])
    yolo_count = len([record for record in records if record.get('detector') == 'yolo'])
    if DETECTION_MODE == 'grid':
        status = f"{turn_label}: grid mode, found {yolo_count} YOLO object(s), {marker_count} ArUco marker(s)"
    else:
        status = f"{turn_label}: full-frame mode, found {yolo_count} YOLO object(s), {marker_count} ArUco marker(s)"
    annotated = draw_detection_label(annotated, status, 20, 30, (255, 255, 255))

    if len(records) < MAX_OBJECTS:
        warning = 'Expected two revealed cards. Check YOLO threshold, ArUco dictionary, grid alignment, lighting, or visibility.'
        annotated = draw_detection_label(annotated, warning, 20, 65, (0, 255, 255))

    return annotated


def print_aruco_marker_report(records):
    marker_records = [record for record in records if 'aruco_id' in record]
    if not marker_records:
        print('No ArUco markers detected')
        return

    print('ArUco markers detected:')
    for record in sorted(marker_records, key=lambda item: (item['row'], item['col'], item['aruco_id'])):
        center_x, center_y = record.get('aruco_center', record['center'])
        print(f"  id={record['aruco_id']} cell={record['cell']} row={record['row'] + 1} col={record['col'] + 1} center=({center_x:.1f}, {center_y:.1f})")


def show_detection_frame(frame, records, turn_label='Current detection'):
    annotated = annotate_records_frame(frame, records, turn_label=turn_label)
    _, encoded_frame = cv2.imencode('.jpeg', annotated)
    display_handle.update(Image(data=encoded_frame.tobytes()))


def debug_panel_image(image, title, target_height=360):
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    if image.dtype != np.uint8:
        scale = 255.0 if np.max(image) <= 1.0 else 1.0
        image = np.clip(image * scale, 0, 255).astype(np.uint8)

    height, width = image.shape[:2]
    resize_scale = target_height / height
    resized = cv2.resize(
        image,
        (max(1, int(round(width * resize_scale))), target_height),
        interpolation=cv2.INTER_NEAREST if resize_scale > 1 else cv2.INTER_AREA,
    )
    panel = cv2.copyMakeBorder(resized, 34, 0, 0, 0, cv2.BORDER_CONSTANT, value=(0, 0, 0))
    cv2.putText(panel, title, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    return panel


def show_debug_panels(panels, display_target=None, target_height=360):
    canvas = np.hstack([debug_panel_image(image, title, target_height=target_height) for title, image in panels])
    _, encoded_frame = cv2.imencode('.jpeg', canvas)
    image = Image(data=encoded_frame.tobytes())
    if display_target is None:
        display(image)
    else:
        display_target.update(image)


def show_debug_one_shot_frame(frame, records, turn_label='Alignment test'):
    raw_frame = draw_grid(frame.copy())
    raw_frame = draw_detection_label(raw_frame, f'{turn_label}: captured photo', 20, 30, (255, 255, 255))
    global TURN_DEBUG_SHOW_ALL_DETECTIONS
    previous_show_all = TURN_DEBUG_SHOW_ALL_DETECTIONS
    TURN_DEBUG_SHOW_ALL_DETECTIONS = True
    try:
        processed_frame = annotate_records_frame(frame, records, turn_label=f'{turn_label}: processed')
    finally:
        TURN_DEBUG_SHOW_ALL_DETECTIONS = previous_show_all
    show_debug_panels([
        ('Captured photo', raw_frame),
        ('Processed detection', processed_frame),
    ])


def turn_score_thresholds(base_threshold=YOLO_SCORE_THRESHOLD, fallback_thresholds=None):
    """Return the base threshold followed by lower fallback thresholds."""
    if fallback_thresholds is None:
        fallback_thresholds = YOLO_FALLBACK_SCORE_THRESHOLDS

    thresholds = [float(base_threshold)]
    for threshold in sorted({float(value) for value in fallback_thresholds}, reverse=True):
        if threshold < base_threshold and not any(np.isclose(threshold, existing) for existing in thresholds):
            thresholds.append(threshold)
    return thresholds


def capture_turn_frames(
    photo_count=TURN_CAPTURE_PHOTO_COUNT,
    settle_seconds=TURN_CAPTURE_SETTLE_SECONDS,
    flush_frames=TURN_CAPTURE_FLUSH_FRAMES,
    interval_seconds=TURN_CAPTURE_PHOTO_INTERVAL_SECONDS,
):
    """Capture several fresh frames for one turn so one bad photo does not decide the turn."""
    frames = []
    for photo_index in range(max(1, int(photo_count))):
        wait_seconds = settle_seconds if photo_index == 0 else interval_seconds
        frames.append(capture_frame(flush_frames=flush_frames, settle_seconds=wait_seconds))
    return frames


def records_with_attempt_metadata(records, photo_number, score_threshold):
    annotated_records = []
    for record in records:
        annotated_record = dict(record)
        annotated_record['source_photo'] = int(photo_number)
        annotated_record['score_threshold'] = float(score_threshold)
        annotated_records.append(annotated_record)
    return annotated_records


def attempt_record_score(records):
    best_records = sorted(records, key=record_selection_key, reverse=True)[:MAX_OBJECTS]
    return sum(record.get('score', 1.0) + (0.25 if 'aruco_id' in record else 0.0) for record in best_records)


def best_attempt(attempts):
    return max(
        attempts,
        key=lambda attempt: (
            min(len(attempt['records']), MAX_OBJECTS),
            attempt_record_score(attempt['records']),
            -attempt['photo_number'],
        ),
    )


def detect_turn_from_frames(frames, fallback_thresholds=None):
    """Try all photos and select the one with the strongest combined detection report."""
    attempts = []
    for threshold in turn_score_thresholds(fallback_thresholds=fallback_thresholds):
        threshold_attempts = []
        for photo_number, frame in enumerate(frames, start=1):
            records = records_for_current_mode(frame, score_thresh=threshold)
            records = records_with_attempt_metadata(records, photo_number, threshold)
            attempt = {
                'photo_number': photo_number,
                'score_threshold': float(threshold),
                'frame': frame,
                'records': records,
            }
            attempts.append(attempt)
            threshold_attempts.append(attempt)

        complete_attempts = [attempt for attempt in threshold_attempts if len(attempt['records']) >= MAX_OBJECTS]
        if complete_attempts:
            return best_attempt(complete_attempts), attempts

    return best_attempt(attempts), attempts


def detect_revealed_cards(
    show=True,
    print_result=True,
    turn_label='Current turn',
    settle_seconds=TURN_CAPTURE_SETTLE_SECONDS,
    flush_frames=TURN_CAPTURE_FLUSH_FRAMES,
    photo_count=TURN_CAPTURE_PHOTO_COUNT,
    photo_interval_seconds=TURN_CAPTURE_PHOTO_INTERVAL_SECONDS,
    fallback_thresholds=None,
    update_memory=True,
    show_debug_one_shot=False,
):
    """Capture several frames, select the best combined detection report, optionally update object_memory, and show debug output."""
    frames = capture_turn_frames(
        photo_count=photo_count,
        settle_seconds=settle_seconds,
        flush_frames=flush_frames,
        interval_seconds=photo_interval_seconds,
    )
    selected_attempt, attempts = detect_turn_from_frames(frames, fallback_thresholds=fallback_thresholds)
    frame = selected_attempt['frame']
    records = selected_attempt['records']
    memory = update_object_memory_from_grid_records(records) if update_memory else None

    if show:
        label = f"{turn_label} photo {selected_attempt['photo_number']}/{len(frames)}"
        if show_debug_one_shot:
            show_debug_one_shot_frame(frame, records, turn_label=label)
        else:
            show_detection_frame(frame, records, turn_label=label)

    if print_result:
        print(f"Detection mode: {DETECTION_MODE}")
        print(f"Card position mode: {CARD_POSITION_MODE}")
        print(f"ArUco dictionary: {ARUCO_DICTIONARY_NAME}")
        print(f"Photos captured: {len(frames)}")
        print(f"Selected photo: {selected_attempt['photo_number']}")
        for attempt in attempts:
            marker_count = len([record for record in attempt['records'] if 'aruco_id' in record])
            yolo_count = len([record for record in attempt['records'] if record.get('detector') == 'yolo'])
            print(f"  photo {attempt['photo_number']} yolo={yolo_count} markers={marker_count}")
        print_aruco_marker_report(records)
        if update_memory:
            print_object_memory(memory)
        else:
            print('Debug only: object_memory was not changed.')
            for object_number, record in enumerate(sorted(records, key=record_selection_key, reverse=True)[:MAX_OBJECTS], start=1):
                marker_text = f" id={record['aruco_id']}" if 'aruco_id' in record else ''
                print(f"debug_{object_number}: {record['description']}{marker_text} at {record['cell']} score={record.get('score', 1.0):.2f}")
        if update_memory and len(memory) < MAX_OBJECTS:
            print('Warning: fewer than two revealed cards were recorded for this turn after all photos.')

    return memory if update_memory else records


def show_current_detection(settle_seconds=0.0):
    """Convenience helper for testing camera alignment without recording a turn."""
    return detect_revealed_cards(
        show=True,
        print_result=True,
        turn_label='Alignment test',
        settle_seconds=settle_seconds,
        photo_count=1,
        update_memory=False,
        show_debug_one_shot=True,
    )


MNIST_MODEL_PATHS = [
    NOTEBOOK_DIR / 'dpu_mnist_classifier.xmodel',
    Path('/home/root/jupyter_notebooks/PYNQ_Bootcamp/bootcamp_sessions/PYNQ 201 - MNIST/dpu_mnist_classifier.xmodel'),
    Path('/home/root/jupyter_notebooks/pynq-dpu/dpu_mnist_classifier.xmodel'),
]
mnist_runner = None
mnist_input_data = None
mnist_output_data = None
mnist_output_size = None
mnist_camera = None
mnist_camera_device = None
paid_hint_state = {
    'active': False,
    'name': '',
    'row': None,
}


def mnist_model_path():
    for path in MNIST_MODEL_PATHS:
        if path.exists():
            return path
    raise FileNotFoundError('Could not find dpu_mnist_classifier.xmodel. Check the PYNQ 201 - MNIST notebook folder.')


def calculate_mnist_softmax(data):
    result = np.exp(data)
    return result / np.sum(result)


def ensure_mnist_runner():
    global mnist_runner, mnist_input_data, mnist_output_data, mnist_output_size
    if mnist_runner is not None:
        return mnist_runner

    overlay.load_model(str(mnist_model_path()))
    mnist_runner = overlay.runner
    input_tensors = mnist_runner.get_input_tensors()
    output_tensors = mnist_runner.get_output_tensors()
    shape_in = tuple(input_tensors[0].dims)
    shape_out = tuple(output_tensors[0].dims)
    mnist_output_size = int(output_tensors[0].get_data_size() / shape_in[0])
    mnist_input_data = [np.empty(shape_in, dtype=np.float32, order='C')]
    mnist_output_data = [np.empty(shape_out, dtype=np.float32, order='C')]
    return mnist_runner


def ensure_yolo_runner():
    global dpu, inputTensors, outputTensors, shapeIn, shapeOut0, shapeOut1, shapeOut2, input_data, output_data, image, mnist_runner
    overlay.load_model(str(MODEL_PATH))
    dpu = overlay.runner
    inputTensors = dpu.get_input_tensors()
    outputTensors = dpu.get_output_tensors()
    shapeIn = tuple(inputTensors[0].dims)
    shapeOut0 = tuple(outputTensors[0].dims)
    shapeOut1 = tuple(outputTensors[1].dims)
    shapeOut2 = tuple(outputTensors[2].dims)
    input_data = [np.empty(shapeIn, dtype=np.float32, order='C')]
    output_data = [
        np.empty(shapeOut0, dtype=np.float32, order='C'),
        np.empty(shapeOut1, dtype=np.float32, order='C'),
        np.empty(shapeOut2, dtype=np.float32, order='C'),
    ]
    image = input_data[0]
    mnist_runner = None
    return dpu


def largest_digit_bounding_box(binary):
    # Picking the single largest external contour is not enough on its own: on a
    # real webcam frame that "largest blob" is very often the sheet of paper, a
    # shadow, or the frame border rather than the digit's ink. Filter those out
    # by area (too small = noise, too large = background/paper) and by rejecting
    # anything that touches the edge of the region we're searching in.
    height, width = binary.shape[:2]
    region_area = height * width
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    candidates = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < 0.01 * region_area or area > 0.85 * region_area:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        if x <= 1 or y <= 1 or x + w >= width - 1 or y + h >= height - 1:
            continue
        candidates.append((area, (x, y, w, h)))

    if not candidates:
        return None
    return max(candidates, key=lambda item: item[0])[1]


def preprocess_mnist_frame(frame):
    # MNIST digits are white-on-black, cropped to the stroke, scaled to a ~20x20
    # box and shifted so the center of mass sits in the middle of a 28x28 image.
    # Reproducing that normalization is what makes webcam captures classify well;
    # simply shrinking the whole frame to 28x28 leaves the digit tiny and grey.

    # Assume the paper is held roughly centered under the camera and restrict the
    # search to the middle of the frame. This keeps desk clutter, hands, and the
    # paper's own outer edge (which usually touches the full frame border) from
    # being picked up as the "digit" blob.
    frame_h, frame_w = frame.shape[:2]
    roi_h, roi_w = int(frame_h * 0.7), int(frame_w * 0.7)
    y0, x0 = (frame_h - roi_h) // 2, (frame_w - roi_w) // 2
    roi = frame[y0:y0 + roi_h, x0:x0 + roi_w]

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Otsu picks the threshold from the image itself, so it copes with changing
    # lighting far better than a fixed cutoff. INV assumes a dark digit on bright
    # paper; if the ROI is already light-on-dark, flip so the digit stays white.
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    if np.mean(binary) > 127:
        binary = 255 - binary

    # Clear out speckle noise so it doesn't create spurious small contours.
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))

    box = largest_digit_bounding_box(binary)
    if box is not None:
        x, y, w, h = box
        pad = max(2, int(0.15 * max(w, h)))
        x0c, y0c = max(0, x - pad), max(0, y - pad)
        x1c, y1c = min(binary.shape[1], x + w + pad), min(binary.shape[0], y + h + pad)
        digit = binary[y0c:y1c, x0c:x1c]
    else:
        # No blob passed the filters (e.g. the digit itself reaches the ROI
        # edge). Fall back to the bounding box of every foreground pixel
        # rather than defaulting to the whole ROI, which would smush a mostly
        # blank region down to 28x28 instead of the digit's actual shape.
        nonzero = cv2.findNonZero(binary)
        if nonzero is not None:
            x, y, w, h = cv2.boundingRect(nonzero)
            digit = binary[y:y + h, x:x + w]
        else:
            digit = binary

    digit_h, digit_w = digit.shape
    if digit_h == 0 or digit_w == 0:
        return np.zeros((28, 28, 1), dtype=np.float32)

    # Scale the crop to fit inside a 20x20 box while preserving aspect ratio -
    # using one scale factor for both axes so the digit isn't stretched/smushed.
    scale = 20.0 / max(digit_h, digit_w)
    new_w = max(1, int(round(digit_w * scale)))
    new_h = max(1, int(round(digit_h * scale)))
    digit = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Paste into a 28x28 black canvas.
    canvas = np.zeros((28, 28), dtype=np.float32)
    y_off = (28 - new_h) // 2
    x_off = (28 - new_w) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = digit

    # Shift so the center of mass lands on the middle pixel, matching MNIST.
    total = canvas.sum()
    if total > 0:
        ys, xs = np.mgrid[0:28, 0:28]
        center_y = (ys * canvas).sum() / total
        center_x = (xs * canvas).sum() / total
        translation = np.float32([[1, 0, 14 - center_x], [0, 1, 14 - center_y]])
        canvas = cv2.warpAffine(canvas, translation, (28, 28), flags=cv2.INTER_NEAREST)

    normalized = np.asarray(canvas / 255.0, dtype=np.float32)
    return np.expand_dims(normalized, axis=2)


def open_mnist_camera(device=None):
    global mnist_camera, mnist_camera_device
    if mnist_camera is not None and mnist_camera.isOpened():
        mnist_camera.release()

    if device is None:
        device = mnist_camera_device
    if device is None:
        candidates = [candidate for candidate in video_devices() if candidate != globals().get('camera_device')]
        device = candidates[0] if candidates else globals().get('camera_device')
    if device is None:
        raise RuntimeError('No MNIST camera device selected.')

    mnist_camera = open_camera(device)
    mnist_camera_device = device
    return mnist_camera


def read_mnist_camera_frame(device=None, read_attempts=8):
    if device is not None or mnist_camera is None or not mnist_camera.isOpened():
        open_mnist_camera(device)

    for _ in range(max(1, int(read_attempts))):
        ret, frame = mnist_camera.read()
        if ret and frame is not None:
            return frame
        mnist_camera.grab()
        ret, frame = mnist_camera.retrieve()
        if ret and frame is not None:
            return frame
        time.sleep(0.05)

    open_mnist_camera(device)
    ret, frame = mnist_camera.read()
    if ret and frame is not None:
        return frame
    raise RuntimeError('Could not read a frame from the MNIST camera.')


def run_mnist_model_from_processed(digit_image):
    runner = ensure_mnist_runner()
    mnist_input_data[0][0, ...] = digit_image.reshape(mnist_input_data[0].shape[1:])
    job_id = runner.execute_async(mnist_input_data, mnist_output_data)
    runner.wait(job_id)
    logits = mnist_output_data[0].reshape(1, mnist_output_size)[0]
    probabilities = calculate_mnist_softmax(logits)
    prediction = int(probabilities.argmax())
    return prediction, probabilities


def predict_mnist_digit_from_frame(frame, show=True):
    digit_image = preprocess_mnist_frame(frame)
    prediction, probabilities = run_mnist_model_from_processed(digit_image)

    if show:
        plt.figure(figsize=(2, 2))
        plt.imshow(digit_image[:, :, 0], 'gray')
        plt.title(f'MNIST prediction: {prediction}')
        plt.axis('off')
        plt.show()

    # Restore YOLO so normal board detection still works after a paid hint capture.
    ensure_yolo_runner()
    return prediction


def capture_mnist_digit(device=None, show=True):
    frame = read_mnist_camera_frame(device=device)
    return predict_mnist_digit_from_frame(frame, show=show)


def mnist_debug_snapshot(device=None):
    frame = read_mnist_camera_frame(device=device)
    digit_image = preprocess_mnist_frame(frame)
    prediction, probabilities = run_mnist_model_from_processed(digit_image)
    return frame, digit_image[:, :, 0], prediction, probabilities


def show_mnist_debug_snapshot(frame, digit_image, prediction, probabilities, display_target=None):
    confidence = float(probabilities[prediction])
    annotated_frame = draw_detection_label(
        frame.copy(),
        f'MNIST captured photo: prediction {prediction} confidence {confidence:.2f}',
        20,
        30,
        (255, 255, 255),
    )
    show_debug_panels([
        ('MNIST captured photo', annotated_frame),
        (f'Processed 28x28 -> {prediction} ({confidence:.2f})', digit_image),
    ], display_target=display_target)


def debug_mnist_one_shot(device=None):
    try:
        frame, digit_image, prediction, probabilities = mnist_debug_snapshot(device=device)
        show_mnist_debug_snapshot(frame, digit_image, prediction, probabilities)
        confidence = float(probabilities[prediction])
        print(f'MNIST debug captured from {device or mnist_camera_device}: prediction {prediction} confidence {confidence:.3f}')
        return {'prediction': prediction, 'confidence': confidence}
    finally:
        ensure_yolo_runner()


def debug_mnist_loop(device=None):
    debug_display_handle = display(None, display_id=True)
    try:
        while True:
            frame, digit_image, prediction, probabilities = mnist_debug_snapshot(device=device)
            show_mnist_debug_snapshot(frame, digit_image, prediction, probabilities, display_target=debug_display_handle)
    except KeyboardInterrupt:
        print('MNIST debug loop stopped.')
    finally:
        ensure_yolo_runner()


def inject_paid_hint(card_name, row, col):
    card_name = str(card_name).strip()
    row = int(row)
    col = int(col)
    if not card_name:
        raise ValueError('Enter a card name before using Paid Hint.')
    if not (1 <= row <= GRID_ROWS and 1 <= col <= GRID_COLS):
        raise ValueError(f'Hint location R{row}C{col} is outside the {GRID_ROWS}x{GRID_COLS} grid.')

    cell = f'R{row}C{col}'
    if cell not in computer_player['matched_cells']:
        computer_player['known_cards'][cell] = card_name
    computer_player['history'].append({
        'turn': computer_player['turn_number'],
        'player': 'hint',
        'action': 'paid_hint',
        'cell': cell,
        'description': card_name,
    })
    print(f'Paid hint stored: {card_name} at {cell}')
    return {'cell': cell, 'description': card_name}


REFEREE_PORT = 9000  # adjust to match the referee's server
REFEREE_PAID_HINT_REQUEST_PATH = '/paid_hint_request'  # adjust to match the referee's server
REFEREE_HOST_OVERRIDE = ''  # set from the widgets' "Referee IP" box


def referee_host():
    return str(REFEREE_HOST_OVERRIDE or '').strip()


def request_paid_hint_from_referee(card_name):
    """POST a paid-hint request (with the card name) out to the referee.

    The referee is expected to respond by sending two MNIST digit photos (the
    row digit, then the column digit) to this board's POST /paid_hint_image.
    """
    host = referee_host()
    if not host:
        print('No Referee IP set (see the widgets) - skipping the outbound paid hint request.')
        return None

    url = f'http://{host}:{REFEREE_PORT}{REFEREE_PAID_HINT_REQUEST_PATH}'
    payload = {'name': card_name}
    try:
        response = requests.post(url, json=payload, timeout=10)
        print(f'Requested paid hint for {card_name!r} from referee at {url} (status {response.status_code}).')
        return response
    except requests.RequestException as exc:
        print(f'Could not reach the referee at {url}: {exc}')
        return None


def start_paid_hint(card_name):
    """Begin a paid hint: remember the card name and ask the referee for the row/column digits."""
    card_name = str(card_name).strip()
    if not card_name:
        raise ValueError('Enter a card name before starting a paid hint.')
    paid_hint_state['active'] = True
    paid_hint_state['name'] = card_name
    paid_hint_state['row'] = None
    request_paid_hint_from_referee(card_name)
    print(f"Paid hint started for '{card_name}'. Waiting for the referee to send row/column digit photos to /paid_hint_image.")
    return dict(paid_hint_state)


def reset_paid_hint_state():
    paid_hint_state['active'] = False
    paid_hint_state['name'] = ''
    paid_hint_state['row'] = None


def _decode_image_payload(image_data):
    """Decode a base64 string or data URL into a BGR OpenCV image."""
    if not image_data:
        raise ValueError("Request body must include an 'image' field (base64 or data URL).")
    if isinstance(image_data, str) and image_data.startswith('data:'):
        image_data = image_data.split(',', 1)[1]
    raw_bytes = base64.b64decode(image_data)
    frame = cv2.imdecode(np.frombuffer(raw_bytes, dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise ValueError('Could not decode image data.')
    return frame


def submit_paid_hint_digit_image(image_data, card_name=None):
    """Feed a photographed handwritten digit into the paid-hint row/column flow.

    Wires the image API into the same flow as the physical MNIST camera capture:
    the first image submitted after a paid hint starts is the row, the second is
    the column, and the hint is then stored exactly like inject_paid_hint().
    """
    if card_name and not paid_hint_state['active']:
        start_paid_hint(card_name)
    if not paid_hint_state['active']:
        raise RuntimeError("No paid hint is active. Include a 'name' field with the first image, or click Paid Hint first.")

    frame = _decode_image_payload(image_data)
    digit = predict_mnist_digit_from_frame(frame, show=False)

    if paid_hint_state['row'] is None:
        paid_hint_state['row'] = digit
        print(f'Paid hint row captured from image: {digit}. Send the column digit image next.')
        return {'stage': 'row', 'digit': digit}

    row = paid_hint_state['row']
    col = digit
    result = inject_paid_hint(paid_hint_state['name'], row, col)
    reset_paid_hint_state()
    return {'stage': 'column', 'digit': digit, **result}


def print_game_memory():
    known_cards = computer_player.get('known_cards', {})
    matched_cells = computer_player.get('matched_cells', set())

    free_hints = computer_player.get('free_hints', {})
    if not known_cards and not matched_cells and not free_hints and not object_memory:
        print('No game memory recorded yet.')
        return computer_player

    if known_cards:
        print('Known unmatched cards:')
        for cell, description in sorted(known_cards.items()):
            print(f'  {cell}: {description}')
    else:
        print('Known unmatched cards: none')

    if matched_cells:
        print('Matched cells:')
        for cell in sorted(matched_cells):
            print(f'  {cell}')
    else:
        print('Matched cells: none')

    if free_hints:
        print('AI riddle hints:')
        for cell, info in sorted(free_hints.items(), key=lambda item: -item[1].get('score', 0.0)):
            description = info.get('description')
            description_text = f" ({description})" if description else ''
            print(f"  {cell}{description_text}: score={info.get('score', 0.0):.3f} source={info.get('source', 'ai_riddle')}")
    else:
        print('AI riddle hints: none')

    if object_memory:
        print('Latest detection memory:')
        print_object_memory(object_memory)

    return computer_player


def parse_cell_name(cell):
    row = int(cell[1:cell.index('C')]) - 1
    col = int(cell[cell.index('C') + 1:]) - 1
    return row, col


def store_free_hints(cell_scores):
    computer_player['free_hints'] = dict(cell_scores)
    computer_player['history'].append({
        'turn': computer_player['turn_number'],
        'player': 'hint',
        'action': 'free_hint',
        'cells': sorted(cell_scores.keys()),
        'scores': {cell: info['score'] for cell, info in cell_scores.items()},
    })
    return computer_player['free_hints']


def clear_free_hints():
    computer_player['free_hints'] = {}
    return computer_player['free_hints']


# ---------------------------------------------------------------------------
# Hint Communications API
#
# Paid hint flow (outbound then inbound):
#   1. start_paid_hint(card_name) [Paid Hint button] POSTs {"name": card_name}
#      out to the referee (see request_paid_hint_from_referee / Referee IP + port).
#   2. The referee responds by sending two MNIST digit photos - row, then
#      column - to this board's POST /paid_hint_image below.
#
# Free hint (riddle) flow (inbound then outbound):
#   - POST /riddle {"riddle": "<string>"} arrives at this board, which then
#     forwards the riddle to the halo Strix LLM (through bootcamp_ai) along
#     with the cards already seen this game. The LLM guesses which already-seen
#     card the riddle describes, and that cell is stored in
#     computer_player['free_hints']. Replaces the old pose-based "free hint".
#
# Inbound endpoints served on this board (one small HTTP server):
#   POST /riddle              {"riddle": "<string>"}
#   POST /paid_hint_image     {"image": "<base64 or data URL>", "name": "<optional>"}
# ---------------------------------------------------------------------------
RIDDLE_HINT_MODEL = 'Smart Helper'  # bootcamp_ai friendly model name used to solve riddles
HINT_API_PORT = 8890
COMMS_IP_OVERRIDE = ''  # set from the widgets' "Comms IP" box; blank = auto-detect
hint_api_server = None
hint_api_server_thread = None


def communications_ip():
    """IP address to tell outside devices to send hint API requests to."""
    override = str(COMMS_IP_OVERRIDE or '').strip()
    return override or bootcamp_ai.board_ip()


def _known_unmatched_cards():
    matched = computer_player['matched_cells']
    return {
        cell: description
        for cell, description in computer_player['known_cards'].items()
        if cell not in matched
    }


def build_riddle_hint_prompt(riddle_text, known_cards):
    if known_cards:
        card_lines = '\n'.join(f'- {cell}: {description}' for cell, description in sorted(known_cards.items()))
    else:
        card_lines = '(no cards have been revealed yet)'

    return (
        'You are the hint helper for a memory-matching card game. Some cards on the board have '
        "already been revealed and are listed below by their grid cell and what they show.\n\n"
        f'Already-seen cards:\n{card_lines}\n\n'
        f'A player just gave this riddle for a hint: "{riddle_text}"\n\n'
        'Work out the single object the riddle describes. If that object matches one of the '
        "already-seen cards above, answer with that card's cell. If it does not match any "
        'already-seen card (or none have been revealed yet), set "cell" to null.\n'
        'Reply with ONLY compact JSON, no extra words, in exactly this shape:\n'
        '{"object": "<short object name>", "cell": "<R#C# or null>"}'
    )


def _extract_json_object(reply_text):
    match = re.search(r'\{.*\}', reply_text, re.DOTALL)
    if not match:
        raise ValueError(f'LLM reply did not contain JSON: {reply_text!r}')
    return json.loads(match.group(0))


def ask_llm_for_riddle_hint(riddle_text, model=None):
    """Send a riddle to the halo Strix LLM and parse back an object + grid cell guess."""
    riddle_text = str(riddle_text).strip()
    if not riddle_text:
        raise ValueError('Riddle text is empty.')

    known_cards = _known_unmatched_cards()
    llm_prompt = build_riddle_hint_prompt(riddle_text, known_cards)

    bootcamp_ai.use_model(model or RIDDLE_HINT_MODEL)
    reply_text = bootcamp_ai.prompt(llm_prompt, enforce_cooldown=False, session='memory_game_riddle_hint')
    parsed = _extract_json_object(reply_text)

    object_name = str(parsed.get('object') or '').strip()
    cell = parsed.get('cell')
    cell = str(cell).strip().upper() if cell else None
    if cell in ('NULL', 'NONE', ''):
        cell = None
    if cell is not None and cell not in known_cards:
        # Only trust cells the LLM was actually told about.
        cell = None

    return {'riddle': riddle_text, 'object': object_name, 'cell': cell, 'raw_reply': reply_text}


def store_riddle_hint(result):
    cell = result.get('cell')
    if cell is None:
        print(f"AI riddle hint: no already-seen card matched \"{result['riddle']}\" (best guess: {result['object']!r}).")
        return computer_player['free_hints']

    row, col = parse_cell_name(cell)
    computer_player['free_hints'][cell] = {
        'score': 1.0,
        'row': row,
        'col': col,
        'source': 'ai_riddle',
        'description': result['object'],
    }
    computer_player['history'].append({
        'turn': computer_player['turn_number'],
        'player': 'hint',
        'action': 'ai_riddle_hint',
        'riddle': result['riddle'],
        'object': result['object'],
        'cell': cell,
    })
    print(f"AI riddle hint: \"{result['riddle']}\" -> {result['object']} at {cell}")
    return computer_player['free_hints']


def request_riddle_hint(riddle_text, model=None):
    """Solve a riddle with the halo Strix LLM and store the resulting hint in game memory."""
    result = ask_llm_for_riddle_hint(riddle_text, model=model)
    store_riddle_hint(result)
    return result


class HintApiRequestHandler(BaseHTTPRequestHandler):
    protocol_version = 'HTTP/1.1'

    def log_message(self, *args):
        pass

    def _send_json(self, code, payload):
        data = json.dumps(payload).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json')
        self.send_header('Content-Length', str(len(data)))
        self.send_header('Access-Control-Allow-Origin', '*')
        self.end_headers()
        self.wfile.write(data)

    def _read_json_body(self):
        content_length = int(self.headers.get('Content-Length', 0))
        return json.loads(self.rfile.read(content_length) or b'{}')

    def do_POST(self):
        path = self.path.rstrip('/')
        try:
            body = self._read_json_body()
        except Exception as exc:
            self._send_json(400, {'error': f'bad request: {exc}'})
            return

        if path == '/riddle':
            riddle_text = str(body.get('riddle') or '').strip()
            if not riddle_text:
                self._send_json(400, {'error': "Request body must include a 'riddle' string."})
                return
            try:
                result = request_riddle_hint(riddle_text, model=body.get('model'))
                self._send_json(200, result)
            except Exception as exc:
                self._send_json(500, {'error': str(exc)})
            return

        if path == '/paid_hint_image':
            image_data = body.get('image')
            if not image_data:
                self._send_json(400, {'error': "Request body must include an 'image' string (base64 or data URL)."})
                return
            try:
                result = submit_paid_hint_digit_image(image_data, card_name=body.get('name'))
                self._send_json(200, result)
            except Exception as exc:
                self._send_json(500, {'error': str(exc)})
            return

        self._send_json(404, {'error': 'not found'})


def start_hint_api_server(port=HINT_API_PORT):
    """Start the small HTTP server for /riddle and /paid_hint_image."""
    global hint_api_server, hint_api_server_thread
    if hint_api_server is not None:
        print(f'Hint API already running on port {hint_api_server.server_address[1]}')
        return hint_api_server

    last_error = None
    for candidate_port in range(port, port + 20):
        try:
            server = ThreadingHTTPServer(('0.0.0.0', candidate_port), HintApiRequestHandler)
        except OSError as exc:
            last_error = exc
            continue
        hint_api_server = server
        hint_api_server_thread = threading.Thread(target=server.serve_forever, daemon=True)
        hint_api_server_thread.start()
        base_url = f'http://{communications_ip()}:{candidate_port}'
        print(f'Hint API listening at {base_url}')
        print('POST JSON like {"riddle": "I have four wheels and an engine"} to ' + base_url + '/riddle')
        print('POST JSON like {"image": "<base64 photo of a digit>", "name": "<card name, first call only>"} to ' + base_url + '/paid_hint_image')
        return server

    raise RuntimeError(f'Could not open a port for the hint API server: {last_error}')


def stop_hint_api_server():
    global hint_api_server, hint_api_server_thread
    if hint_api_server is not None:
        hint_api_server.shutdown()
        hint_api_server.server_close()
    hint_api_server = None
    hint_api_server_thread = None
    print('Hint API stopped.')


## 8. Computer Opponent Turn Functions

These functions let the notebook act as the computer player, not as a helper for the human player.

Use this flow:

1. Human flips two cards, the camera detects them, then call `record_opponent_turn()`.
2. Call `start_computer_turn()` to let the computer choose two grid cells.
3. Flip the computer's chosen cells on the physical board, let the camera detect them, then call `record_computer_turn_result()`.

The computer only uses cells it has learned from previous revealed turns. If it does not know a matching pair, it explores unknown cells.

In [13]:
computer_player = {
    'known_cards': {},          # cell -> description
    'free_hints': {},           # cell -> {'score', 'row', 'col', 'source'}
    'matched_cells': set(),
    'computer_score': 0,
    'opponent_score': 0,
    'turn_number': 0,
    'pending_computer_cells': (),
    'history': [],
}


def all_grid_cells():
    return [f'R{row + 1}C{col + 1}' for row in range(GRID_ROWS) for col in range(GRID_COLS)]


def reset_computer_player():
    computer_player['known_cards'] = {}
    computer_player['free_hints'] = {}
    computer_player['matched_cells'] = set()
    computer_player['computer_score'] = 0
    computer_player['opponent_score'] = 0
    computer_player['turn_number'] = 0
    computer_player['pending_computer_cells'] = ()
    computer_player['history'] = []
    return computer_player


def clear_game_memory():
    empty_object_memory()
    return reset_computer_player()


def revealed_cards_from_memory(memory=None, turn_label='Current turn'):
    memory = detect_revealed_cards(turn_label=turn_label) if memory is None else memory
    revealed = []

    for info in memory.values():
        cell = info.get('cell')
        description = info.get('description')
        if cell and description:
            revealed.append({'cell': cell, 'description': description})

    by_cell = {}
    for card in revealed:
        by_cell[card['cell']] = card['description']

    return [{'cell': cell, 'description': description} for cell, description in by_cell.items()]


def remember_revealed_cards(revealed_cards):
    for card in revealed_cards:
        cell = card['cell']
        description = card['description']
        if cell not in computer_player['matched_cells']:
            computer_player['known_cards'][cell] = description


def matching_revealed_pair(revealed_cards):
    if len(revealed_cards) < 2:
        return None

    first, second = revealed_cards[:2]
    if first['cell'] != second['cell'] and first['description'] == second['description']:
        return (first['cell'], second['cell'])

    return None


def mark_match(cells):
    for cell in cells:
        computer_player['matched_cells'].add(cell)
        computer_player['known_cards'].pop(cell, None)
        computer_player['free_hints'].pop(cell, None)


def known_matching_pair():
    descriptions = {}

    for cell, description in computer_player['known_cards'].items():
        if cell in computer_player['matched_cells']:
            continue
        descriptions.setdefault(description, []).append(cell)

    for cells in descriptions.values():
        available_cells = [cell for cell in cells if cell not in computer_player['matched_cells']]
        if len(available_cells) >= 2:
            return tuple(sorted(available_cells[:2]))

    return None


def unknown_cells():
    known = set(computer_player['known_cards'])
    matched = computer_player['matched_cells']
    return [cell for cell in all_grid_cells() if cell not in known and cell not in matched]


def available_free_hint_cells():
    matched = computer_player['matched_cells']
    known = set(computer_player['known_cards'])
    return {
        cell: info for cell, info in computer_player.get('free_hints', {}).items()
        if cell not in matched and cell not in known
    }


def choose_free_hint_cells():
    hints = available_free_hint_cells()
    if len(hints) < 2:
        return None

    rows = {}
    for cell, info in hints.items():
        row = info.get('row')
        if row is None:
            row, _ = parse_cell_name(cell)
        rows.setdefault(row, []).append((cell, float(info.get('score', 0.0))))

    best_row = max(
        rows.items(),
        key=lambda item: (len(item[1]), sum(score for _, score in item[1])),
    )
    row_cells = sorted(best_row[1], key=lambda item: -item[1])
    if len(row_cells) >= 2:
        return tuple(sorted([row_cells[0][0], row_cells[1][0]])), 'free_hint_line'

    top_cells = sorted(hints.items(), key=lambda item: -item[1].get('score', 0.0))[:2]
    return tuple(sorted([top_cells[0][0], top_cells[1][0]])), 'free_hint_candidates'


def choose_exploration_cells():
    cells = unknown_cells()
    random.shuffle(cells)

    choices = cells[:2]
    if len(choices) < 2:
        fallback = [
            cell for cell in all_grid_cells()
            if cell not in computer_player['matched_cells'] and cell not in choices
        ]
        random.shuffle(fallback)
        choices.extend(fallback[:2 - len(choices)])

    return tuple(choices)


def start_computer_turn():
    computer_player['turn_number'] += 1

    move = known_matching_pair()
    reason = 'known_match'

    if move is None:
        free_move = choose_free_hint_cells()
        if free_move is not None:
            move, reason = free_move
        else:
            move = choose_exploration_cells()
            reason = 'explore'

    computer_player['pending_computer_cells'] = move
    computer_player['history'].append({
        'turn': computer_player['turn_number'],
        'player': 'computer',
        'action': 'start_turn',
        'cells': move,
        'reason': reason,
    })

    print(f"Computer turn {computer_player['turn_number']}: flip {move[0]} and {move[1]} ({reason})")
    return move


def record_computer_turn_result(memory=None):
    revealed = revealed_cards_from_memory(memory, turn_label='Computer turn result')
    remember_revealed_cards(revealed)

    pair = matching_revealed_pair(revealed)
    matched = pair is not None
    if matched:
        mark_match(pair)
        computer_player['computer_score'] += 1

    computer_player['history'].append({
        'turn': computer_player['turn_number'],
        'player': 'computer',
        'action': 'record_result',
        'chosen_cells': computer_player['pending_computer_cells'],
        'revealed': revealed,
        'matched': matched,
    })
    computer_player['pending_computer_cells'] = ()

    print(f"Computer revealed: {revealed}")
    print(f"Computer match: {matched}")
    return {'revealed': revealed, 'matched': matched, 'score': computer_score()}


def record_opponent_turn(memory=None):
    revealed = revealed_cards_from_memory(memory, turn_label='Opponent turn result')
    remember_revealed_cards(revealed)

    pair = matching_revealed_pair(revealed)
    matched = pair is not None
    if matched:
        mark_match(pair)
        computer_player['opponent_score'] += 1

    computer_player['history'].append({
        'turn': computer_player['turn_number'],
        'player': 'opponent',
        'action': 'record_result',
        'revealed': revealed,
        'matched': matched,
    })

    print(f"Opponent revealed: {revealed}")
    print(f"Opponent match: {matched}")
    return {'revealed': revealed, 'matched': matched, 'score': computer_score()}


def computer_score():
    return {
        'computer': computer_player['computer_score'],
        'opponent': computer_player['opponent_score'],
        'matched_pairs': len(computer_player['matched_cells']) // 2,
    }


def computer_player_status():
    return {
        'score': computer_score(),
        'known_cards': dict(sorted(computer_player['known_cards'].items())),
        'free_hints': dict(sorted(computer_player.get('free_hints', {}).items())),
        'matched_cells': sorted(computer_player['matched_cells']),
        'pending_computer_cells': computer_player['pending_computer_cells'],
    }


## 9. Example Game Flow and Debug Start

There is no always-running camera loop. Each record function captures one fresh frame and uses the selected detection mode to process the currently revealed cards. The turn calls also show a debug camera image with the grid, YOLO boxes, class labels, confidence scores, and grid cells.

Recommended debug startup flow:

```python
reset_computer_player()
set_detection_mode('grid')        # or set_detection_mode('full_frame')
camera_test_loop()
show_current_detection()
computer_player_status()
```

For the human player's turn:

```python
# Human flips two cards on the physical board.
record_opponent_turn()
```

For the computer player's turn:

```python
cells_to_flip = start_computer_turn()
# Flip those two cells on the physical board.
record_computer_turn_result()
```

If no objects are recorded, run:

```python
camera_test_loop(run_detection=True)
```

Then adjust camera position, lighting, `BOARD_CORNERS`, or `YOLO_SCORE_THRESHOLD`.

If a turn still appears one move behind, increase `TURN_CAPTURE_SETTLE_SECONDS` or `TURN_CAPTURE_FLUSH_FRAMES`. For example:

```python
TURN_CAPTURE_SETTLE_SECONDS = 0.8
TURN_CAPTURE_FLUSH_FRAMES = 20
```

If cards are still too small, increase `GRID_CROP_SIZE` after rerunning the helper cell. The default is:

```python
GRID_CROP_SIZE = (416, 416)
```

Switch modes any time with:

```python
set_detection_mode('grid')
set_detection_mode('full_frame')
```

In [14]:
# Debug game start helper calls. Run these one at a time while setting up the board.
# reset_computer_player()
# set_detection_mode('grid')        # Small cards: crop/stretch each grid square.
# set_detection_mode('full_frame')  # Original behavior: whole-frame YOLO.
# camera_test_loop(run_detection=True)
# show_current_detection()
# TURN_CAPTURE_SETTLE_SECONDS = 0.8
# TURN_CAPTURE_FLUSH_FRAMES = 20
# GRID_CROP_SIZE = (416, 416)
# computer_player_status()
#show_current_detection()
# First human turn example after two cards are physically flipped:
# record_opponent_turn()

# First computer turn example:
# cells_to_flip = start_computer_turn()
# print('Computer chose:', cells_to_flip)
# After flipping those physical cells:
# record_computer_turn_result()


## 10. Release Camera

In [15]:
cap.release()

In [16]:
# Memory Game Widget Controls
import traceback
import ipywidgets as widgets
from IPython.display import clear_output, display


detection_approach_dropdown = widgets.ToggleButtons(
    options=[
        ('YOLO full frame', 'yolo_full_frame'),
        ('YOLO grid crops', 'yolo_grid_crops'),
        ('ArUco border',    'aruco_border'),
        ('ArUco per-card',  'aruco_per_card'),
        ('ArUco border + grid crops', 'aruco_border_grid_crops'),
    ],
    value=globals().get('DETECTION_APPROACH', 'aruco_per_card'),
    description='Approach:',
    style={'description_width': 'initial'},
)
def available_camera_options():
    devices = video_devices()
    remembered_devices = [
        globals().get('camera_device'),
        globals().get('mnist_camera_device'),
    ]
    for device in remembered_devices:
        if device and device not in devices:
            devices.append(device)
    return devices


camera_options = available_camera_options()
board_camera_dropdown = widgets.Dropdown(
    options=camera_options,
    value=globals().get('camera_device') if globals().get('camera_device') in camera_options else (camera_options[0] if camera_options else None),
    description='Board cam:',
)
mnist_camera_default = next(
    (device for device in camera_options if device != board_camera_dropdown.value),
    board_camera_dropdown.value,
)
mnist_camera_dropdown = widgets.Dropdown(
    options=camera_options,
    value=mnist_camera_default,
    description='MNIST cam:',
)
camera_flip_checkbox = widgets.Checkbox(
    value=globals().get('CAMERA_VERTICAL_FLIP', False),
    description='Flip board cam vertically',
    indent=False,
)
comms_ip_text = widgets.Text(
    value=globals().get('COMMS_IP_OVERRIDE', ''),
    placeholder='auto-detect, e.g. 192.168.1.42',
    description='Comms IP:',
)
referee_ip_text = widgets.Text(
    value=globals().get('REFEREE_HOST_OVERRIDE', ''),
    placeholder='e.g. 192.168.1.50',
    description='Referee IP:',
)
referee_port_text = widgets.IntText(
    value=globals().get('REFEREE_PORT', 9000),
    description='Referee port:',
)
alignment_detection_checkbox = widgets.Checkbox(
    value=True,
    description='Show detections in alignment loop',
    indent=False,
)
settle_slider = widgets.FloatSlider(
    value=TURN_CAPTURE_SETTLE_SECONDS,
    min=0.0,
    max=2.0,
    step=0.1,
    description='Settle sec:',
    readout_format='.1f',
)
photo_count_slider = widgets.IntSlider(
    value=TURN_CAPTURE_PHOTO_COUNT,
    min=1,
    max=8,
    step=1,
    description='Photos:',
)
flush_slider = widgets.IntSlider(
    value=TURN_CAPTURE_FLUSH_FRAMES,
    min=0,
    max=40,
    step=1,
    description='Flush:',
)
grid_rows_slider = widgets.IntSlider(
    value=GRID_ROWS,
    min=1,
    max=12,
    step=1,
    description='Rows:',
)
grid_cols_slider = widgets.IntSlider(
    value=GRID_COLS,
    min=1,
    max=12,
    step=1,
    description='Cols:',
)

gui_output = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})


def apply_widget_settings():
    global GRID_ROWS, GRID_COLS, TURN_CAPTURE_SETTLE_SECONDS, TURN_CAPTURE_PHOTO_COUNT, TURN_CAPTURE_FLUSH_FRAMES, camera_device, COMMS_IP_OVERRIDE, REFEREE_HOST_OVERRIDE, REFEREE_PORT
    set_detection_approach(detection_approach_dropdown.value)
    if board_camera_dropdown.value and board_camera_dropdown.value != globals().get('camera_device'):
        camera_device = board_camera_dropdown.value
        reopen_camera()
    set_camera_vertical_flip(camera_flip_checkbox.value)
    GRID_ROWS = int(grid_rows_slider.value)
    GRID_COLS = int(grid_cols_slider.value)
    TURN_CAPTURE_SETTLE_SECONDS = float(settle_slider.value)
    TURN_CAPTURE_PHOTO_COUNT = int(photo_count_slider.value)
    TURN_CAPTURE_FLUSH_FRAMES = int(flush_slider.value)
    COMMS_IP_OVERRIDE = comms_ip_text.value.strip()
    REFEREE_HOST_OVERRIDE = referee_ip_text.value.strip()
    REFEREE_PORT = int(referee_port_text.value)


def refresh_camera_dropdowns():
    camera_options = available_camera_options()
    previous_board_camera = board_camera_dropdown.value
    previous_mnist_camera = mnist_camera_dropdown.value

    board_camera_dropdown.options = camera_options
    mnist_camera_dropdown.options = camera_options

    board_camera_dropdown.value = (
        previous_board_camera
        if previous_board_camera in camera_options
        else globals().get('camera_device')
        if globals().get('camera_device') in camera_options
        else camera_options[0]
        if camera_options
        else None
    )
    mnist_camera_dropdown.value = (
        previous_mnist_camera
        if previous_mnist_camera in camera_options
        else next((device for device in camera_options if device != board_camera_dropdown.value), board_camera_dropdown.value)
        if camera_options
        else None
    )

    print(f'Available cameras: {camera_options if camera_options else "none found"}')
    return camera_options


def run_widget_action(label, action):
    with gui_output:
        clear_output(wait=True)
        print(label)
        try:
            apply_widget_settings()
            result = action()
            if result is not None:
                print(result)
        except Exception:
            traceback.print_exc()


def make_button(description, action, style='', tooltip=''):
    button = widgets.Button(description=description, button_style=style, tooltip=tooltip)
    button.on_click(lambda _button: run_widget_action(description, action))
    return button


def reset_paid_hint_capture_button():
    reset_paid_hint_state()
    paid_hint_capture_button.description = 'Capture Row'
    paid_hint_capture_button.disabled = True
    paid_hint_capture_button.layout.display = 'none'


def start_paid_hint_flow():
    start_paid_hint(paid_hint_name_text.value)
    paid_hint_capture_button.description = 'Capture Row'
    paid_hint_capture_button.disabled = False
    paid_hint_capture_button.layout.display = ''


def capture_paid_hint_digit():
    if not paid_hint_state['active']:
        raise RuntimeError('Click Paid Hint first.')

    digit = capture_mnist_digit(device=mnist_camera_dropdown.value, show=True)
    if paid_hint_state['row'] is None:
        paid_hint_state['row'] = digit
        paid_hint_capture_button.description = 'Capture Column'
        print(f"Captured row: {digit}. Write the column digit, then click Capture Column.")
        return {'row': digit}

    row = paid_hint_state['row']
    col = digit
    result = inject_paid_hint(paid_hint_state['name'], row, col)
    reset_paid_hint_capture_button()
    return result


reset_button = make_button('Reset Game', reset_computer_player, 'warning')
status_button = make_button('Show Status', computer_player_status, 'info')
memory_button = make_button('Show Memory', print_game_memory, 'info')
clear_memory_button = make_button('Clear Memory', clear_game_memory, 'warning')
refresh_cameras_button = make_button('Refresh Cameras', refresh_camera_dropdowns, 'info')

alignment_loop_button = make_button(
    'Start Alignment Loop',
    lambda: camera_test_loop(run_detection=alignment_detection_checkbox.value),
    'primary',
    'Runs until you interrupt/stop the kernel.',
)
reopen_camera_button = make_button('Reopen Camera', reopen_camera, 'warning')
current_detection_button = make_button('Debug One-Shot', show_current_detection, 'primary')
detect_cards_button = make_button(
    'Debug Detect Cards',
    lambda: detect_revealed_cards(turn_label='Debug detection', update_memory=False),
    'primary',
)
mnist_debug_button = make_button(
    'Debug MNIST One-Shot',
    lambda: debug_mnist_one_shot(device=mnist_camera_dropdown.value),
    'primary',
)
mnist_loop_button = make_button(
    'Debug MNIST Loop',
    lambda: debug_mnist_loop(device=mnist_camera_dropdown.value),
    'primary',
    'Runs until you interrupt/stop the kernel.',
)

opponent_turn_button = make_button('Record Opponent Turn', record_opponent_turn, 'success')
computer_start_button = make_button('Start Computer Turn', start_computer_turn, 'success')
computer_result_button = make_button('Record Computer Result', record_computer_turn_result, 'success')
paid_hint_name_text = widgets.Text(
    value='',
    placeholder='card name, e.g. car',
    description='Hint name:',
)
paid_hint_button = make_button('Paid Hint', start_paid_hint_flow, 'warning')
paid_hint_capture_button = make_button('Capture Row', capture_paid_hint_digit, 'warning')
paid_hint_capture_button.disabled = True
paid_hint_capture_button.layout.display = 'none'

riddle_hint_text = widgets.Text(
    value='',
    placeholder='riddle, e.g. I have four wheels and an engine',
    description='Riddle:',
)
riddle_hint_button = make_button(
    'Ask Riddle Hint',
    lambda: request_riddle_hint(riddle_hint_text.value),
    'info',
)
start_hint_api_button = make_button('Start Hint API Server', start_hint_api_server, 'info')
clear_ai_hints_button = make_button('Clear AI Hints', clear_free_hints, 'warning')


controls = widgets.VBox([
    widgets.HTML('<h3>Memory Game Controls</h3>'),
    widgets.HTML('<b>Modes</b>'),
    detection_approach_dropdown,
    widgets.HTML('<b>Cameras</b>'),
    widgets.HBox([board_camera_dropdown, mnist_camera_dropdown, refresh_cameras_button]),
    widgets.HBox([camera_flip_checkbox]),
    widgets.HTML('<b>Network</b>'),
    widgets.HTML('<small>Comms IP: what outside devices use to reach this board\'s Hint API below (blank = auto-detect this board\'s wifi IP). Referee IP/port: where this board sends outbound paid-hint requests.</small>'),
    widgets.HBox([comms_ip_text, referee_ip_text, referee_port_text]),
    widgets.HTML('<b>Board Grid</b>'),
    widgets.HBox([grid_rows_slider, grid_cols_slider]),
    widgets.HBox([settle_slider, photo_count_slider, flush_slider]),
    widgets.HTML('<b>Testing and Alignment</b>'),
    widgets.HTML('<small>Debug only: these buttons do not update object memory or game state.</small>'),
    alignment_detection_checkbox,
    widgets.HBox([reopen_camera_button, alignment_loop_button, current_detection_button, detect_cards_button]),
    widgets.HTML('<b>Model Debug</b>'),
    widgets.HTML('<small>One-shot captures one frame. Loops run continuously until you interrupt/stop the kernel.</small>'),
    widgets.HBox([mnist_debug_button, mnist_loop_button]),
    widgets.HTML('<b>Gameplay</b>'),
    widgets.HBox([reset_button, status_button, memory_button, clear_memory_button]),
    widgets.HBox([opponent_turn_button, computer_start_button, computer_result_button]),
    widgets.HTML('<b>Paid Hint</b>'),
    widgets.HTML('<small>Type a card name and click Paid Hint: the board POSTs the name to the referee (Referee IP/port above), and the referee sends back two digit photos to /paid_hint_image (row, then column). You can also capture the digits locally with the MNIST camera and Capture Row/Column instead.</small>'),
    widgets.HBox([paid_hint_name_text, paid_hint_button, paid_hint_capture_button]),
    widgets.HTML('<b>AI Riddle Hint / Hint API</b>'),
    widgets.HTML('<small>Type a riddle and click Ask Riddle Hint, or click Start Hint API Server so an outside app can POST {"riddle": "..."} to /riddle or a digit photo to /paid_hint_image at the Comms IP above. The halo Strix LLM guesses the riddle\'s object and matches it to an already-seen card.</small>'),
    widgets.HBox([riddle_hint_text, riddle_hint_button, start_hint_api_button, clear_ai_hints_button]),
    widgets.HTML('<small>Tip: the alignment loop is continuous. Use the notebook stop/interrupt control to stop it.</small>'),
    gui_output,
])

display(controls)